# Deney 04 — Trend / Fırsat Skorlama (+ YouTube Data API v3 zenginleştirme)

**Amaç:** `ContentOpportunity` için trend_score/revenue_score/competition_level hesaplama yaklaşımlarını karşılaştırmak: (A) saf istatistiksel/heuristik yöntem, (B) LLM tabanlı yorumlayıcı skorlama (aday sağlayıcılar), (C) kota-verimli YouTube Data API v3 gerçek video/kanal metrikleri.

**Sağlayıcı bağımsızlığı politikası (Faz 7 düzeltmesi):** `TMDB_API_KEY`, `YOUTUBE_DATA_API_KEY` ve LLM anahtarlarının hepsi **birbirinden bağımsız**dır — hiçbiri zorunlu değildir, hiçbirinin eksikliği notebook'u çökertmez:
- TMDB yoksa → TMDB/Trends/LLM bölümleri temiz atlanır, YouTube bölümü yine de çalışabilir.
- YouTube yoksa → YouTube bölümü temiz atlanır, TMDB/Trends/LLM yine de çalışır.
- İkisi de yoksa → geçerli boş/`insufficient_data` durumlu artefaktlar yazılır ve ne gerektiği açıkça yazdırılır.
- LLM adayı verisi yoksa (TMDB atlandıysa) LLM skorlama da atlanır — boş veriyle sahte bir sonuç üretilmez.

API anahtarları (Colab 🔑 Secrets): `TMDB_API_KEY` (opsiyonel, ücretsiz kayıt), `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY` (LLM adayları için, opsiyonel), `YOUTUBE_DATA_API_KEY` (opsiyonel). `pytrends` (Google Trends) anahtar gerektirmez.

Bu notebook'un ürettiği normalize edilmiş YouTube veri seti Notebook 05'in gerçek geçmiş veriyle çalışması için girdi sağlar — sıfır satır olsa bile her zaman sabit bir sütun şemasıyla yazılır (bkz. bölüm 4).


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Bu notebook'ta kullanilan gorev: **toplu siniflama** -> Rota B (`gemini-3.5-flash-lite` -> `gemini-3.1-flash-lite` -> `gemma-4-26b-a4b-it`), `thinking_level='minimal'`.


In [ ]:
!pip -q install requests pytrends pandas openai anthropic google-genai google-api-python-client > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, time, json, hashlib
from pathlib import Path
import requests
import pandas as pd

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
YOUTUBE_DATA_DIR = RESULTS_DIR / 'youtube_data'
YOUTUBE_DATA_RAW_DIR = YOUTUBE_DATA_DIR / 'raw'
YOUTUBE_DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

# Faz 4/5/6 ile ayni sema: Notebook 05'in guvenle okuyabilmesi icin normalize
# edilmis YouTube veri setinin sutunlari HER ZAMAN sabittir (0 satir olsa bile).
NORMALIZED_VIDEO_COLUMNS = [
    'video_id', 'title', 'description', 'published_at', 'channel_id', 'channel_title',
    'view_count', 'like_count', 'comment_count', 'duration_iso8601', 'tags', 'category_id',
]

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

TMDB_API_KEY = get_secret('TMDB_API_KEY')
OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
GOOGLE_API_KEY = get_secret('GeminiKey') or get_secret('GOOGLE_API_KEY')  # Colab Secret adi: GeminiKey (eski ad: GOOGLE_API_KEY)
YOUTUBE_DATA_API_KEY = get_secret('YoutubeKEY') or get_secret('YOUTUBE_DATA_API_KEY')  # Colab Secret adi: YoutubeKEY (eski ad: YOUTUBE_DATA_API_KEY)
# gercek anahtar degeri asla notebook icine yazilmaz/yazdirilmaz
print('TMDB:', 'var' if TMDB_API_KEY else 'yok (TMDB/Trends/LLM bolumleri atlanacak)')
print('OpenAI:', 'var' if OPENAI_API_KEY else 'yok')
print('Anthropic:', 'var' if ANTHROPIC_API_KEY else 'yok')
print('Google:', 'var' if GOOGLE_API_KEY else 'yok')
print('YouTube Data API:', 'var' if YOUTUBE_DATA_API_KEY else 'yok (YouTube zenginlestirmesi atlanacak)')

if not TMDB_API_KEY and not YOUTUBE_DATA_API_KEY:
    print('\nUYARI: Ne TMDB_API_KEY ne YOUTUBE_DATA_API_KEY var. Bu calistirma sadece bos/')
    print('insufficient_data statuslu artefaktlar uretecek. En az birini eklemek icin:')
    print(' - TMDB_API_KEY: https://www.themoviedb.org/settings/api (ucretsiz)')
    print(' - YOUTUBE_DATA_API_KEY: https://console.cloud.google.com/ -> YouTube Data API v3 (ucretsiz kota)')

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

results = []
def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({'task': task, 'candidate': candidate, 'quality_score_1_5': quality_score,
                     'cost_usd_estimate': round(cost_usd, 5), 'latency_sec': round(latency_sec, 2),
                     'notes': notes, 'status': status})
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} status={status}")

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['youtube_data', 'google_ai']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'youtube_data': {
        'sha256': '048809136491091e9650d1783a6708cf1bc59c4a719983c19ccb1fb85b4a9f69',
        'encoding': 'zlib+b85',
        'data': (
            'c-q~4*>>Ybvfw+vB8T_(6;P%qmF@EMuuS`^CH1w&wRK75ZnvyPfkaY53k29$q++S{5p&MG&o9m=Of0zqpr~@YUdHFRL;$%)M#h#A'
            'k;CEet!%1zmdTMQWt{95dA1fm7tL8Slj10@V{!22MBIEO=0%ptd7T!y$coE!J{}BCui{cBVzw6hUq^K*^CT)(tD>x%JgwJJH809E'
            'zZ|Elbv_%3{b$ilTE<zlf=Vg1q(8-ZGT7gb)<si8w^7{G#WKbM(K4=TS#mjXlg1(zv!bZ0x{Oz1F0)LexsdVvO5{Z?;nTGUDk%r='
            'vto{2r7M}Gx!luDo61p@r>m8$$E)>wu_#JWUr8Z9#Pizepq3x%L46h1BCUj6&Sa9neAB!xpm<S9(`glost_>@r@kts6!Vxy2IXGA'
            'Iv%9URVJ4*uW3&39_AD$qFCtC#VVDRppNE6lO-Z4(?u;{%vh*e=LBszZ%TkG6v<(9Fwh_=a0CFCB8zJ&Zm%Q&BS~w3I<y93UZj}}'
            'Cj%jN#Th;?i=@e5*z!h}wB8i}w372u)?!f>OK}J-%|uhlGQqV6xD?e`=CE)ua2$JGL2E)3w>S^z38r)-pVNe8S&3C#RRl`JnrpeH'
            '<@vD6(s^0~VgW$j&zdw#qIs4AK;DmOY_AF!EsXg}maqsowiRGbU5yac;9nTV3R+LAnj0_UTipRS@V<(1Fdt<k%d#jVT)6jQ&w1aJ'
            '*?aNw-RYSCypC5eB3MZnPMoh{fOEORrJ<=|UEB=Ss>or9XGOB65xjf*0_p;)NPq$&#4W5w4G_mx#Q!{f{c7*<^d|y26sM+W7BU$@'
            'y?G2k6_-VEnaOyS^7L)b6+jRnI?tL!(i|5Jpin9Q(a5T%#<2h3_9_KjM+}Goji?|(ImUH_qU9P^pp2IiHcdG0tULTNn|y+^U_`*h'
            'g9c!w@?=%OkX=-Ss_9}qq9HV^tcVj{k7kyo)fK{z*A9?Fk&W?SI2;ZLouX&~`>vExBw!N&Wq|U4v9u4W!9cycimR(Eo$1HFSFjT5'
            'H;{oV{ky32uTtt?b-I+?72vDHC*6^L9#NS@W_8RzKNh*FvVz8;UtQx3_BA*?J2-oH8vS(e;@$D7n2I}^`4Es;mMLxGY&a6b1R$3c'
            'D|`{HvUo19iY$>Oek$cnDsLlPyA(*>kYP6jq$`UXd=^7kyJoR~O@Y;Mr{T**45&k&p<vPwg^X~$*3iWLVDReo+m{C~PX2m)6#aB^'
            'bo@FxeEs6x%U3wabME3MO=J<J)J6>(PVz{rd6^QHQQN$5GKu0EpXXPwBr;RA^pmQsn}z8BHsn+xORuGUniorgfq9WMC6n&7Dt`aX'
            '{!{NAkQ&`+9bZ<|0IbtxQLfbh61j+*td6oczii@5$wg@NQG5dsLSCf`5&^<laf`6qZsIxc9l)=K+D@yex`L&R%m84QB`Kl?kDXYW'
            '0>$8BFgQ9nJ$(Js@!K8{Z-o9A8v1~%bA&7Gy(M+8U(m+15UONc&aa}P1bm@#ylgPp=&ES&l?8hh0ou`PEGY9!C`Irt)f^BD=bGEf'
            'Rw4C{n=9p9=5wc@Uh7h>WL(?#%*oATlETj$oIx4q*Bv+={&4W>)$xliGCqh{7C1b41DqBR6|Mlu$RZyNm+^<F3!$Zq^T?ZkUYifm'
            'Bj)vQ;uL5w6JX%BCDLdu4E>{lQ3%}J6=2dx@G_2JGa;#j4ZQ>&4zz`tKN1*d4)`VUN`=<q>HhvMSCrLW60gOqnO}nx0%RJv26W48'
            '*-+w2H;pYrdnj2kr^X>(%3Oj(1+*>&=E?)q4gM(Q3Yb@Y#p5BkRRRY;Lr#!GV}HU5XCO*k%ksGq3P3!oB=G=X7bHdrU_t2;B@E;<'
            '4TwI#`H1n42Y-6^`s@G}9~djpO1;b#fn_)mPm$te@fsgbYPs}2(9(Nv9GK#p`@zxi9}nKWIEzk?58fXB5FH-8coDrgd3kacy*Yjx'
            '9Uc4(y#h=PLjD{<iPwJyynOxk==d$7V=&bF9ns}b<4-9wc@x88(?g6h(=bE~_(6i01rweNl7)A74F;SdEdfC=OE1$|nxQO~z~0Fd'
            'Tz7!Y3W$jyLW2@5t`H&=R|1NgE%p!;vK*1rRu!3q_MxVxPGe@)C=^!6P6Tg;O0`6kdoI46ekGP^-qdD5x;_ZRFe>21GY}jPVOzdF'
            'L*X275c(FxeLzegs3U5k%s?%OK#niR@T8g+w<$3V3BnDtd@PJ(Ruw?BIQ`-E+q3BC-P?n+lh?1JACBL>Jvlu)IgDN&`~_C)5ZC4R'
            '`-6cd4<|>_n}f5n<F~KiODV^|H&(#zf^zt`cZcV{-0oi7eYrpS^6UG5hgFD9PG2MMu<iHOrL7GOya~Ra1mC~;7WPy1-7m=(;ZgYg'
            '+4t%tkyLv7L#W>|`1tKh^A?y>0c(kG;mt2&{Q7-(8h-!p!vSChfNb}-`wv<>1mOTSS_&HsS(e1T2f)Pz40to*9>0mRMyfRof8#L>'
            'fGi1?4?s7XoAD3}l>v1firw#EeC31*DJqO0(Shu_1JGKemmu9GJ17zEEYqrj!SCRfd36k%k;$l?CABw3;gA-N+CeU?n;D99q2>!D'
            'L;}_37Kfh!_{XRxfQEolwNesz55eG3Jy>g5pJaFvaE%gzPg?GUrbt)@l6aZs0JVh4Gf2~@p%?)lt81NN{97+$^yBf*(VyNS`FnHl'
            '_Tc5FtrmQrhQG`L&=~$%7uPcXXFQ+Fs*3o5^}ui%od4}VeyM&L?p%BkB1t4H=~1u()|5b-iJ-*^B8m7X=yJdUr}U?TI;S-Vq7?#W'
            '7i#QQSMdtq<>)PyCY%DOE3r1zZWtwkbV2P@4Qp?q0W=M1&w?5laRnlfudRV5c?q3q&5T!4CgYAy_CsB-7&}xutecVgzfm(A3D(S%'
            '23LvqF!1TO<MH^r_lQ!&OqWenTiqK<g9vhsh*8xE==x@7AY1{2067A2NH1Y)Xh_quQxN}*fq0-A5~xDJ)Sf(f0s`8>;o0#K{2YdE'
            '+nX?ETs5;Gi<h$`7R!lP4DXiX%d%)z!PD@5+G=qm?C{iXCW}c6V?gx|;D17p9%`}#wq!KfX<Lg`i_c5l)xi;%uBwgD<dtsIncrx}'
            'xF|50@Ut)kMnIJK>XP-p&q45A15DuV-q5Vdp&kOt2<;7`#UV5b(wnh(8>ba4;!n6%ym!=QRn6m4E#j<Ni$Gh7K*B-u0?0S>>&-k!'
            'EYec#GNcUEb-IF1LC6OJ4MNWqF-h2+bGE&qRVJ;`7>LLe8XSPlL*h$&1C=;#XV}5;=hyGf-u>YiL`x8BflwjZh;?!O+yfi*cUh*{'
            'GG4}6I#yeYaI^)ivC%TLm4TWWxR^<Af}t`|lA!t({HrVsviuRYh1u0}pc^tWtxo?4P?efU&$Ad9=3*UTaiWR=P{q!k`vVZJUAb7`'
            'a8#9a&fKq9^enigh-A`?*-jP{f}%Puv<{EG08bpD0Wl$~D&TU?R^wd|xN4wy@nv41bX{3;S2`=w3q+bqr6FjBGcdJaeL=ASoJD~i'
            '=RC5dLfbg853&vb3A~i>t0oAIku#(Ep)Ci*y}W8buq%MiRH#8466_AXQy5KIV-%n(>IUt0gcKlYCNdZ{^<ww;!!R7fSb`zV5<0)T'
            'cVFOe&hPFo!t?zLe$A~To(%mlG6xjLnh|yfhUQuCB>+wXWGcSi-|w3Xi)m>NlsHWy#rTE~HS=_PGay5NDh+Z+f_VL4YhQI+gAxkH'
            'psfT$(tL5+Ks?xrgbMMCbeY!fTeO+@FT-xr?qI3iNb1tthQZR*tD-)%A}!kWigK1FK-RW(hr1TYR3vFv5sZ|829?vk0qB~xi-!RD'
            '_2y;4ZWo{$Cz~1|>dQaBn~G--VM--SAWckvxVg=UX%Mn!2M33Sc+f^3iL-PmflBhWc7SPSA?w?sypGV43eflUj3&R%i`!g}3slqB'
            'pqU<$dU<wydKMl3<;{zOSEO$`hF_(F5SC6vKY!p;lT(tPtDQPVh9Pd`I<}`$vV`e@-U))sM?e+;5U>xKq+@YfWa9t+pa1>802VnO'
            'R>3L)wBxHfE>m%hXFX6r)=&jFg>5jMLo++C^dlq6j;Wca;^1(s&0iaK9n1%n6w}+KoTFU)k;k5{YP9_%FwnFbi~k~`3_?oLBmxD5'
            '0-k`gKns=l514&<lfoGAmDOx3(R_rdZkp8C8ppSwK7zjIR1zSvDvQ5EchK5(nrCRLBGAH^1ujdPrR66L{zW<~W8h`*bEl`v9sF=i'
            '!!)CB(Fqvqe>Hj5T*FMEUR)6bozeU#0j2n3(v;1z;nn<vL6YeXgs6DEfadd9%;IYi%$hPz;_DP-hf7(4m<0uZ*GN&vSAcs-T83uK'
            'cE`hTPe2vQVTrC`jWPgg*5n}CAa?#pZ%&SOpcfe8yF*ZE1RinC;wpm@p!6nj&Ikj7Sdz&dD3%nLWh2({N04lE5s8+iBC`W%ZB|6a'
            '%UCoS!i#!`3B{LL42UE?!V5I9!4}#(dUd)7i~a$`jiPMw#sD@PixXh5CA2JbO&qpGxFp3x-D_B108uVG`$ID~1(IvKBLbz7faU?}'
            'J`5z(u(?djBxUFT8OC$#ycfkSjW(^9bfAnd62ThrcaO=0nwVWIkEA+V0Z|gRRJ=eC<6r@g#4Lq%hwAf24HW<cnrHJIzr-JLE&5>K'
            '(?{s!5Ar&b`BJU{F|f@mIuTY)#h@9dglnLqF4K=#GOa=`1d4O~cZHQ>KD;9N6ebl}i7-NmFwiISEb95@1XM7qI!g0ZQ(N^Bod=98'
            '`$m=s^$HK#LG!!aofl}2cg@gjQ6~)nxmc;}S7^(xnv9hTY;!DLk#25uc=sF|WZ9U`ofV3ZDhf-EWncy?M}6mJ4F}-`WXW=3Px<UO'
            'oFkAtnkAI5Q+CSu$0rjKP<uY7N%z}TTt#ffts-rThf<_CIK|kmqb2!nGkA&*_)Z6I?#KIw6)b|qdAs!s6r^3w+O6Pphov84*5_VR'
            '7j3b?S!PdKFF^_~6S9#3rM&?ZWGHq}(peEbLm-`H<k}6jh8DUy34(-sIUlM{lHtXqrC4l+&<3gO*TKj(b|JnHi=hbacx<Fq>`?zZ'
            ';eFWds|`bU1PZNbu*G$E6@9e5AXqav)SHbp(!?w7zV6iN#c-?*l2I1uCfX&at?&V4X-lnm`UUUo@)@;vx=HhX%s$W$|I!wqHPkJQ'
            'wqSx!=`XsY)TX;c7eQ-ej{%s(^bX?tko*j0GR6f#JCZkVL+cYLECF!^>GglIJc2A^9_K}#;!&n9!dUTH4ph=Avgpc(D3*C4N=`iM'
            '58`O|lx$3Ae_VTvW|dVChT6^pJ@DMNw(ix3zj0YtXdw%R|7lG7+sGeqJ%<HvCV`ty06c#EmOEp9J@-~Jh{X=cmpj5=NDx?d)MD<0'
            'qAmu!`Pxeg(pgMiYv?EFO91LT%{g8M@l73Mu`!EA`5aGUl{_lmYl{6It(U|uNq%@w8{j>;wUHCn?bVu1{89}J4V+}0{;(qV67&#L'
            '%!W*q*b170(!7yuujG?|J!W1&$GREM)VUx^8@E>liQj-8M&w{4`P43zUh;v*=p2DWOB<#J9f>OzrKN*D!9sf$bp(Vy`fBP_&5SL)'
            'rSjk~UtIr>uF#=m(YCnZ@$s~G%Y_{_y8;OKxINYohrNMAMj^kR4%1|$yhBIIVs`|nm1fll`5pNNpj9qSrrgt5evtE~mVu>#pvZT&'
            '>E|C2?8qOe!-ob|$@VyyKUz}17#g*QlpC$rh=A3<YpWBo;-@L?j5LOHjYd~-XHe_EmG>Kj3hddj;-Z5H%VR4xb<!I&fL?UuD*Cj2'
            'CT)%O*NzJsqaM%qF9z-YtY`<N&dNr1O0=XJZ0UU1l4U%(NEKozsHJyJ!(1%vRPPmX&<Vyj>zSr~Y;t=jCHfR6Z&X9xH`R;Nzx{;>'
            'S^2|vQ54$f47x)cNZ!D9R0HNad+&5xaEVB(GJSIEZ58XfXtYiC>99iYOjM&&AG57CbK7bn5b)<f$uB;g>DI7D@|+q#&IT>m-xpnO'
            '9Mfk!{_%KnWw@jWG?qs^pNi=*tF!tYaMdH3H0bXQ$Wx*Bk*+4}9uNVY>WN~`K6Z27Ft@-=Lvip&5Pw{$4q5AM0sMKfkmQiZBhyYx'
            'ISL8LQI)G5HO2HOWY-Ke0zqE;5*BkQ%`DLcG+tHo5g`cem+P@O&{6<9yXW+cr4%Je)WwPo1jM>4KDU}lat)~g^;flb1kErQDR80l'
            'pv)bXg&XGlVnqK}fVJvC5q(B9I-s$^!HrMXVjWKZFxq~C^2*a0$1C)L3$#q_me6v}z&kVPp^7lQHc<MhyFst0SVVC~4${$P65Kn8'
            'c2DCtNrJv%dAbqx&^KAD@p#Mp8FZhM)7QJ`8E1X|KtQP259v2oWUy(e{kV+}bzr@@mDaHUZcr;4WDRpznW751W9DOGwS#xwoPGWB'
            '=~t&9+SBo}a>E-_Lz?;!c}o|(fw?w1)UeqHb%Z9T(b91=Pz>c|Lv9IeF93J^oiilth~AO@g`P5-d=Zs~=C)=OHMct|wz(e^@NoN6'
            'G4?z`BCSXOx-_C3gAgWJu!0Q$?*PwX()(P9TpZh_*p=KZro0~81=VY>s9J+|QT5s@s^&ujeIkDU&DZ<;;tMsTC*rGb=uvn5M0|sf'
            'svWD%Zt~I(+{gi=;kHrDIG=)2`Qm4!-UidYor<lVswpd6xy17(Jo!QQS@O^|F&adQmFq0B3XCE0-GV`TCI4ToyICJ#?WN1UzkB2e'
            'EI_>~aLH@*2G#}5DN|BN^0!s;$_hixP(v-z!FUG{pPJl(#fdMiB5O8@nLU(LwjOs6xG=>c7M6YZsn^%7^a!Ya2HJOo-g4ys0>G||'
            'Wjd!2ELn}j+1ZQT_%<#<p#-X2HmiCU<FIzCcp-mt8xPs~#32sEpx}Y1D9R5V3wKQz6$XW|lOGS8bbS~r`7m-cRKlNjQDRl9Z9Kmw'
            '>Ufpbdr0B-O1VIf@KE{HRA^1WkPY}<Hn8qVk?$a{-mQ=`aU1~0YvmfLF+Sk-3eW985GVIVY7+ag;yH^;mj%Ac<d_v8J`+h~CUFBJ'
            'R4eqoLlnhc0k@uOn5$W$42AKGupA7PwVD_YnPSy&UhGo<D}-L~(20(x^A>l)e>lhv<tmnpQi@@)@*M9CvVGPV1zA+uAu{wVDaa8p'
            'CRMK+&k?Cm6k#|~6b0yLZT+mu3B`pux;uU!o6kTZ@q1uigc}Num)GzwQc{3VHANX^1bZ8GwBni`TQ#*JbvFb=z@wx65ucez{h$`n'
            'Jga4h)XdTRvVZT)5aZp{^fZbplOB)K_yHoJ#aY=&|C@N1r~hbV4%ozDlpOhhva#`r4RotcgCQ_DfD(o7I<3vi^{Or|%XoDKE1jg5'
            'c%)_|eRQ#&*TRVpVSASjlC3cC*lEq60qYe{f4^^3HqtvVS{8+L7C5kvXZyRLHn@#C%~9M3>l?roVig;lComTb|0$X(Ta)IHVmNT&'
            'pdn4$X$kQZBzpKa$M9FCr5r7(On5sLS4wQesg6b_&0_4dhA~3KDedg`YM=mf+E*~PMfw2=`Zeme$O@^ct7aMJyC^23aMT)H4x_G~'
            'D`#5mnoO~=&8naSZ45b53vb7w7Hq0RH^;CdUq1ULSPbtp4&49pVSoS5;KBWH42qub^>!sca6^uL3h4oPjFRg*`sS9y^S>SJ{x#nH'
            'xWD^9qVetp`cy?j7st*g-|Tme{aX8UPq>4N?sP}+lLv7>MsGcbBB1%;5xWH_PMvY&-oz7y9h%{UO;Ah(ulOonQ*aH+d&rn|<Q3Az'
            'fP%SA%UVdm|1Q%TnTI5yP|PV)HdFLDBhD>)ww8`$Ft$aehP#toC}Rxg;ME6d14wDLBLeHgsVxHJ(^Ibjl*$0KtIbSKTWo{Z{_t2A'
            'J?Kl=u$kkK%aU)60yeD^)22T9(N;;Z;21<k#NMFo+iB?Lxu%pC;)|(x+GpkSV9;^nWV1YwxBxpuVmm}U01SrhJJmRh%xuy1gp)bD'
            '?z$eJ!J?%Y6AnZ%?jjwlJVq-5=v!=W4RkZ0lIp^)M*6V3rwfY?W9Wg(uZJC~We%RwD)}-cov`TeR<uT%34?4J=%DOzQ=^P;Arp={'
            '@KdP^K?s<K3dMN8?WAZM|IXQsopo{NXqjivG&fDvZGgCPtFEeJ7mi}PFDy-|9!pcOBJp7#lApL|97McxkZ2+{FTmL1Ur3&`Faatb'
            '-`m}?Tt#Xmv>kD?M5HZ!pzSi|-{fX6hQfCR8Dm#z5`_1T`PObS-6HgO`^b$(J1c3HE3;2Vg5X+V!hyJ&r|DE3W0C)1J^j-5Y~@3z'
            'Yy3Av&Zq>?BIf|wH?L1m{z6pw&(KnFTYU!Qh8+5g@c(&K7B}A4XfE_znGj#4K7gh{5mako@oSYeN>~wy`!*ScqW>RdS?C~SmG2@i'
            'NS`I&t6NyYWt_;6f}QbWvz(z+hI+y==#DQB&W=y1rCD6DV3ic}1|!*#S%PAvBAGD_NL&*qrHl`?li~t<QHsQKYOBwwa;MS`lq;NN'
            '%3RrtYz8f>Dd&JYxd}0%ZfXYKc(d-%RTVJ125i7<5XXt3UZd(_1E=r%GzNn3gOyD0Y>AAppFY~sLq+Vk&UMiXO;zYM9i7;f-w$Cc'
            'KkeyJjr7yH)zj<iTYuX!0E4X&aR`Pxc8I-q1i@w*I5WP!5KkOHQcu?-@ieqxwxvVcXI(^Om@|D@zXqNhWp&W2-GZeT**?;?GiQ)@'
            '9&JNsEj(fmI$UWuYpi$g*})GlPhK6JynXIvBZ${!D$rdu-laIDDkWdziu`ba!gd=)3AmMc0NPI}!9Yfp(Q7nmswQi+|MK1?XddvD'
            'Jeb*mxu@lS(WlvOF&z0&BKStLb)*N4oJy~eTEJEncGX52LU|*wjILyHqFu?N8&}6H3{`{zGSsEM+6dWhL1Bl=1VY0$)h*sUR7odB'
            'q0p8n!!Q=8y){3=_HmSQ8Kd{|Wuzwo?JnaFUEJFrZSN1{H+M!`mQQc@2i8y*aVp<`4yQ5_jz-MCcA3g24)ZGU*r)s)rz}b=Bl1nZ'
            'Cp;K6mWju<x5<s&_@WKU=}0>=M~he)&K?NlHqyciOm+t9-v(Pxn)4GmQjzVF^ht_a)Y#DK7o_?iCfQ*(E-$GZ9pYG9NhcuDZ6-t`'
            '8acV3S#qRUQ{_rAYJ@6v7<naf$~jAZvF^A%GL#F*6qM!^GDAr|lmLWtGs>(|=coLm#kvf196NzIA5t}Yl_r|OxWj67%N-8FZMm=^'
            '8%hh<`aGIVO{PH0Q>fK9V5Z+KPqf-JQQdy~bd+9xhRyqq%n`dFpV7M6RE3{ozrylHJ#IFg^Y%yVWqZ=RY4>^-167&<y>VY}-Nw%9'
            'yR>KEStitotqWkKWfwc>(WW8UiEv>3<Z)hZ9cQF8ncChQ2B4#%2iEn-^&O)e8@>Kx{k~%xj=Z$L*Lizj7w28?s0-tJh3dXR6oVi-'
            'HCy#-@Wes0M<5Dk#zNEiMDWgh6yiMC=9VNM#!-oZY9{NrFp-CgI`O`goN`1pVcx<sn|Ao9DwBC&nhM3Y?#vvE2-Ne8FoqD31K^zl'
            'y!&W%hvo0RNQDPCL(hij$!FzJzpb0XZVa)irxqNYZQ}K$_exJtckkvgm6NFhf)N7oEiu7r`fPth@H@lar>lDOkhtb;I^XBdEE2Gh'
            '8l7X<!Lhq-sNf0OA5qsLzsfY<yk|pil?~4L<#Xusjh@7?$6jo}>rH0{+RnA-Dz7JTKCyVF_i{wX1+beuSKd}Q5mOCQ#`QJ~&ekib'
            'qL^*EiLjprahEw7;WG3(R3ihp5MWt89tk&nA$sX-h>!!N>qepj0`6IiU~`Yv)DqC?>c$L*WpT?#KkQswi7Sd@6qpIJ+EW=ID-qa}'
            '<na^oxKzi#BA(H?1|Dg!!wS_8$)tt}SlRtud-E((nI&W!#00qd#8wMi${BXEgDm={+_6n%yDzF!SYfMIyw5$h)qd0Mpy`%$vtlBt'
            'QHda^ivFf>*YmWPZgs+fFP@m?`1qT2pc7|+@EE$mhsv2>r$ctVi=AR_646#+_mf-2OElW5W4;~KUL7yoz^kFYoFu*FduO3Oo^>YU'
            'B$AvsJKJ*_)|moBx1y8!bVEffVJf02S#e6WW6+C;_5!B>M6Jh}jF(DvLkFtR=05h4l=68e&f>LGvQ9jF9QBmMM6DAK6<wUc)U9ol'
            ';dHJ*wB6-#a=W${M}2B7Xea$^2ifTR3;mh)SYL^QCP@oh%o|iYo9x8jkE#H)J)L4U)J6{d5$e3Nt$NNEZy)$+X(x(msExoMPx+f$'
            '?V;SYlgInaxf6HO-A+2}L&w)x;guOhlrZ)CHG*rOAh=o3<kE!f9VvD6C=rz0+YRUYqh$9Em*J_&pxB|kKQQM}EtVKJ{%l{I904a='
            '0Xr(l`7h3_LX0_+$RBXN7MBeGJD<yN#OVTcbQY}&>2@jy3>h=f=L_w9#;x0Nx<a=>O2Cu>rwowt>>(A=d{t*|)oxk#486U8?m_Z{'
            '$9l{zid(%Z1T`z4<M;?R5ycC#<9U851CSPdk$eOIb7YvAa+$S~l+(<_ZSo7Kc+o4c52NAu??svi6~ZwIs8$&MEAqOVX4Vtbd-1JT'
            'OrK3@9326joA$L>9jIlOxG!d_QTpw0q~<#dA1`k|N_cL0&%<(S*Y%B(Wml#QeG|^-Is5!ku5XJn+2vzDn=vi0H&U=onPgi6BL2EE'
            'iFz@CGi<=m63rV~XxWU>l$|=De;vx!rwW_eiQQlreLzP<?LZG~klY%vVdQM*7jCl05~I+DLT`||40NS%T&=PcnhU+aPyCO!6&4!L'
            '0lteOO9D?xqIOXd$C6|n;YbW0f9yOo7E!+~7ZRCMB!%%oP<Uf2YjujHT1~DZpioXIv-Y2&<<@EaU7v5I+HsAIW*kpDqYjhJzBUL='
            '6*Q?G1BQP^PqQ!f_b2=N!_b2nw>GXnrum}i`cQivs<Ou2fa**=;@qBssuEYaRM0K{>lMprndW%DL(|pxOH9_mV1Dj`yK6R6NSzh8'
            'BM?C^kpVyls`ZfU4f;X1M#a9JVx4YtPsjVF%y(j+QvvUf`4l`t?Bv!iTnszsF2pXjiGD%>)mM*LQ{_>%wN;i<KCH6lIcl{f>R6k%'
            '7I&;r&=PqDdaF8yI?D9#JUg1vqV~R^;ek&ml<m&;+?9yb{4YuRD-ziw8R;@A)wvO?Hx?z8gJHsDFi#7#&DRYK9ySGz27MF|eIL<F'
            'oi5j*gy+&bttJDWo0x~AEI<xZ9_Vjk%B-vc8d74Vn0YcYmW2uBfyIkl>N4gAO-BAWC+ot%p3&186KC94vHEk)*2E`7bbyOV*oy`*'
            '2v@{C(6pZ83YTedHQn5lhW>$!ql@7Y=Kry&mULK&8mvN|JNm1gr|9^Xf`&c>Pe-)h(3Xd~qJu)AB3df7vKRl4vI?dnGNu#z=^vDY'
            's9d@l9}pQVO-tjJrBT5b`rD`cA0UQ9ffmj?MXfz@IN^yo9}yo<2@t3_S>$Zx9&&eqOY&jbXsgGMhwx3Mlrd#(_09(AX+@PWN+mbe'
            't~qE+w8l@Z;v8aCzmFQqyI7-gVgVj^QsbW(y1;XNPP)=|`_<!*Z@6JC_C9&S^LqGz#!y&a(f9;0mi><xsVqH<8{IGBhL!7{u#~Cd'
            '1cRf=p_OmamK%}rG-R&Rc5~G&`<&idzm;U9UxdBpWzBf-9ugx`>Wfem|Ix=cPW-kFN|fX-K}FaUe9j)DwisxCleXmKlX(hBi})TW'
            '3aR6$S(k$Qy7XHRu_+Nc8yQrdWgxeC#UU*tbyrB0UgnthF`#^4xp?v7C1yuTK}KbAeu@(pw-Wd>aye9Zg}%i`JbU&#I>RR&`^){`'
            ';a<_#*@&n7kwc#pFM@{>mr%s>vZyfjfpm#ICd=sOf}u!H_kTAQFO_{+-J@akGj0k7<%B3Txv>u^A3(8)XC(z5Cq5GZXroUv{sDyE'
            'tb&IR3v?@(8QrJn?GJWC#2u2@z<P}<{3rBMp*Ty+0anz2+A0s&ltS%=fOs%@IDERK8MBuTHSZkMXgZIfvprHio+bDuV6;X#(zWX+'
            'PV8+AjE#W1Ya8F>8BY7!4uLlDQ^#GLE(}7ltw-UQg6$CKT*EEp*nMp&bME03^i8;--><o-NX=}2q{c(Rv@ioRb*sE_o-VrursU12'
            'a^kyEZ9sz_nNfxIE)B=K#PtmKu9AX=JALh`;Su4(ABax;%azM*?1wV{w22gf#(}sxln8yQ%{uQ?D7%jehbGgqx=#yj%XCKoMp|oK'
            ')^X{k`?PvnFjsADX#xk?tjLJk^2hYArDFN0?XK5%-Pc6PWM(M-U10m+21h|PVkKC$dum@8Hi3~hz<A7?k@za?)bZ}h(pCJ|J@wjm'
            'j_Hc6_apI*>!Ifm1Vz?N-ULv{m=TzBayD>I^;sENLtwuHR_n@6&OMt&#NBuH+{n|@9S561uPY68eQt|ULO$;!XV!CkUR+;fbNd=v'
            'IFRBg)%qOoEgWaX<<k{T_`D7Mi?DB&%1On6Y`a-%tGyx@5445afHxmSds=g3ofrc?)bYvmsjcsztg6j(>cjWmv2>dy$uCdjhmkHT'
            'bI{gsI)SM*>E#Iuoc;#z@+gt)Fcn{lCr?DMYwPZb;LESY6Y<|2ZsXs9*T49^ST<sk-GQ$ojMLR}y4?pLw%3NTGZ|GoXpQp=uX!BE'
            'm(~%T<SV)s4JPYb#`nk1p215GZO>^TF1&JIK*=w$x&mkC<5O&=g{YevdddyqNPBGAn`v(AYC}V@dIk?2z;qv`UVbb6W8c`0e|e&A'
            'L?hd#TD><c+RC()o9eo^73wPV<d?ly#MF>k<bHu`{hg{aj1rXs-nnku7bDS2p+=&IWuvGvbdM=Zl#DCM;p~pK#}@7U`hoq8Ar})D'
            '4bIzFGLnJ~KU_6R*VgH@;Ipel8OOp0)u_5yMLDr%+9xBO9wqWFrL0Ve4Qdf3YiThqHeRK<X#1&ojLXVJ8(9~MY#Z5ETOg;A;GHHk'
            'B6Q4E1lQwScm3E&hx1Ok{!-e-qG#wGcM&V3plot}(MD}0M9~3Auh2mAIM+>Fc#3tW6gPLFsfaT`Uup~aL`7{FJj?P`Ee0Yior0of'
            'iW6VLMJ!Xi5Klu}L-p3sFaNDJ*O5rr!L~5mvdFYwY0R9I$C;?Rp+lulc-Yt)<;9@yT<{_MPz|`bv9=&etA<K!xIt$2vw(VV8$B$E'
            'D{0@yg&rZljf3qgWhbOrk@ud3tE(P90<2+Yterb*H7Q3_=qzKa?BlxZvX0IrxQ2gVBphK8iVTRo>Zz}h){VLDz<X=mTfc&BOsIV_'
            'ectI!bEVEyS_tHcRbCcI)j&VOK7xZO`hb7PAv93-b>baic(IMmVZ~p^;?yZS-OZ^=GYBw7OSuyKX2(=r&z!6%u4hmx!jxWy1%AEZ'
            'xM?be?@osgpEFHS4jVdWVbCn+rzFGWcs#z?a8Ga8IToYcLBg*d+LzC4A@`)F&EIk1e7^q7`Jx4fKB++C&_?OtVUz0SBRnYb&2Mt<'
            'w$<p{bqhXHs8z9yUu(<hu9~G1U{s1KWhJy}wV8?1eZ_Q<uyw{LD76)aNUj+uh+iXheK1*qIf;)x=emm7-j&5OnPuvZO|{x+iM`B<'
            '8J+d%Junt0It`}^xj;I(Z1BFo*7+QH8+Ed5Y<G^*#;9%1AK}2WJP}d{be9Nv7vO15pgzI~HG=D7IeRISs(#K!Ya}ZC3op#!kB>cQ'
            'yfBw(d5!-D=Z}7a`XI8s>8xoJ&m^};r<x;6w-F7X-^#G=A3f?OcPV*5BC|RUkkm~b>CB^nFdPp<tb>U14FQt%R|OBeu>n_-&|}2F'
            'V+QW~_Vldr1;-t@fVk*A^sx$ux);$%u_}l!F>y4y<B{zQM>yG>_l6JX`n@9D|9YK4jnAFa=V<fMuN$z`A@8Hksiez%Mm_0T+RSI<'
            '9T<20>3lnc2ft227$btA{svI&WT$=50Pg5xgUybwq@uNyBQZ3VVEw{3FYn)XS`7OKW)&rL(dv3{IR#apT#|8kQ&HcP<-u)hqHT-9'
            '{ThhAIf>lBTNxB+GTKpLqaQ-&B7v>9nD}+Ib;UirrZeyM606V+<y<`NbDthOZtIw?dNzO>BulC8fpo>EF^UW*&OP0D#;k@k>R>9e'
            '>D!Sj%Y=T5{G%ffYYld*RLliGQ7+a(hkDm*@6iprvfSta>CO0}xnho%0SBGSVuzUc4|G8cmrhIfDo!pEnJZuY9ljXe8JFoQycaPr'
            '6s61ENI~CpullQVV?ok9Z{^frO<z>kMcDvF_;?B2JWJ9NmjZAb?=Ay6xRe#<?kIt)r^A-bm2Rf6c!TpM=f0M?%FY}bHhFL(N(Z%Z'
            '?n6pC0xCIOKBuN27V$O6UyC%Wkw%vo2ZQ%ZC==Np&-Xgc@Op-!9iw7y-T?iq;C77aQnk$ktDB_}Z1f<@%O=C?C}%MyNX6SOiY#gL'
            '2;6HjekCyYzDp3SGY~ZBPMEni3_3Sb3^zn?wc<ra?Jue)Q$RhtPFU#*FBFwgS&&5r)3`HZuV4)GxO6?a7E&fvgsDIxU9absI4e+d'
            'zv{t@GQm8!^(Q78^BmLo=69U>UBBUwPhZ8*6VudLt1r5B%=kuvK!b)OPl#bxfS@8Bslsp3uxDZm{DfS(SEez?!9Cyl>@Th?$Vpke'
            'SI7ko@l|a*)VO2uvXs#&Y{*Z$|Ba+m&tfxeZJ%jdb~0q$IB+@4h~50u)cXAVh!uAn+f@D<6lilwrGwNIpTbQ=wR{I<^-rfLI@7D-'
            'ygHs`aIjVQLAJ*tAnfvv45+QieKx2e=U8o7L0hplgRW&=F}+SaWMvKdRr@w2{Kh~0fG-vKVU|>>*8fUc3(g;+!j&J+ZDAv>IV}^n'
            '@TmL+TU!2?g37wLsMvBBPJ}gqj`HHv#fG`Kks!<h$ihg9pz<<&dwlx(#ZSjatxM`KwY=gFP}g3mqFZiVWu`2Tw8kOkRHk1#NCA`B'
            'R1ICN{9MH5va>4?7EIcCHWDbqRG~7~vZFMJ7!TcCW@6L)*bAF}z&T-eE}!hZwx}D)?_|O7LoK}aH|1yfL>1N_X3LPgF~`F;Epyvc'
            'Oiabd++9UNgEn;OivEpOJs{n&<J}3Jd>5`Jvmrsr;PF6eCzj~J`%i(D=M}I)je4~9)F0iq`MOHlHOAxj5w}E{mD+warn2UDJviG@'
            'n48>h+qWuNi$VJ;CAvn*W=m*?nqf)>@6=PNXfzNPOMEX$?Euj7hVG9Q>%DG=Yza51>#*0<o>Q0uRrvH7;tXHYJ<TBh*D2=1;+qA^'
            '>T^U5Qq$IYtR<spUfocJT)tIc)Z?Mh%A1LIOJuirxeDKP0W=#>dWLsf0N#Thb9nlb2(Dy|<~>|Xl|($mkeJ-2+<4DD(Os5!JJlK^'
            '4uIm{<`sH9FDdyWTAJCgewEfLbR<vWYOJg@$5;_>DIwFby;EdKmsxOG+$bw`d0w)>__t(?RsM)#y;0La+Gv(}HQkhRBy?1BGHrOR'
            'Y^^=exsu21oc`F$7m~%eF3civ;@}S)p`kZ%#7oPI&{@y+el|00hlx7DjE{#N(<nWA^_M_J^l~$xbkWZU7Y0km*7U0|{?Cl+HWIiU'
            'cc3?c;7u@6cebN<KOr~Wi(>Gzf64SvM&>Z3i$)u!rp2Ak<m@TpBc_M8EPF5BwmEK{jkS6?^U)Uz_s$3-lf2>gBpv@1$kIuj|Co&4'
            'Hk&tJAFwr(H&0~Nq#62&=!AcuiRS4->NHtw8BwV+Ju``Yr%V_+pQdBJZ~`+`nnb(ytfeUh#axZp_fg%D*z?ITExQ#E2O`S2U@|?C'
            '<YWU@Yz8R2;D_c-chtL<qJo$mWi~hE&4%I^TMmiis*-ru2phkE)1RIloV`1ZemZ#Z?)Vf1)IX$Is!k0xr0@90{r}%qE%(v2AoODi'
            '*^zp`cy;@5h^qE|aOc>AU{J5`b=!zw3Oi43t{a1i%96(qUNRjYGZ8keR~{7m@RjrEY(!+0pju%rWd~Nj-8fVS+tF@#O?x-EQAxdl'
            '*3&_&Q)--YQoZ}O8Dt4XIB1Dy8*JcMcB8m4Pv<v%;Z1ygXx&0dy6G!;;Rf8BPaUn}j#1X$q76c{dbl6-mc!$C8$M-f;5m+c{W%gV'
            'BWH21a}795P~D^;_QYY4#j{;x8oNg&X5M#S!HV(Q&yw3i8&Xzf@prU+5n=>*nK%xBO_g-%TErQMqXdVd-|JdsQPhfqK4glbx{+T='
            'dHU=rudY2Rv-fgMQ{FrH>Fv8W2jb_09}iAX_I_r?_jO$?DR%E>f7tcv(YFloefF5zh~Qt$u+wUy{c0jQGSaFva$I`jxGGTr4(>A='
            'IKJXpERj$Ez6S`2a4cSO2#-qkPNS!`=n^Wr$w&j0&U4xjrQl1tY6GQ+vAS~(_zy?$Q&SpY=mxrVg<->V1qlft#GVLi_%_AdYe0in'
            'DfU#JurOO2smh<%lv%{Uxn6$b;2&_Npi6<d?3f!+!oHeyCFm`;Gid41=EImq6|}H~uX~5pDJ6Z^(s&o;?4-PlaUXR&zbetxzZkyz'
            '`QYuz#9LrN?FyNsc;J|%BDjO<_u+G~1|d1yyOi^Eo*Li-wxPvJT_UhX^-&$J<CT=|K;48=6#GsW>~*(~bDn9oY;RQqIei1Tcev-4'
            '^qV&Va4UBL*1&^|*b^e0t))H!CDU(RnWVk#dP?JOx7e`5a{YSr?!j7}t?bnHzInOs++{(|>pr_F7>LQrH3g!2fEe{mcjV4t)Hk%y'
            'zwoF!hqf;stA3aqI#5wD^##z54aOLwbz#Sf$(#ER+ZWApCJO($bmJPSub{+u^FVub5$A6uf>Pa&p?fdy3EiVe);u~qLjNhgxJ}b2'
            'gmS_;_Ps&3t;^e@MeLaM-a>cqVebfsUO1($xak5mlZsxI8?-YJ{XZ!unkx'
        ),
    },
    'google_ai': {
        'sha256': 'e82125022c0237de8bc42f08ee360f4edd0c6264a5cb32a28d46ece49c8cb0e6',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}dw<(Fb};<EKLx6Jo?4udWIM@ZXPwo~tJq4S#+R%uPj=Sh(v(EmtSC}TQg)QA``O=f0YHEkS;-`Qw*7T}?M5a65I8tEH=IkO'
            '(Kzs;I10vI*b<qS&f4C+m-ym!9N&b#=nX{kwLcG{K<o(kG4~vi#LLW2#57Jsb{nK(k;MPukF(C!*84ZVis75Vxi~&Oyg2HM{$B^@'
            '!}G0n<$qf*eHg;$G3N1P37_-t{A4AfIP*twJQJg35N0CCQW4)r;`+vqyr6Rs27Z(^JDrYmExc&5b?YUQaXj%S;u?nW!|wJC&A9!v'
            '^SnI`z4UhbS_DxTM1DsMZ@o;+y@?OQieSEo(=-^tm$O(jVQsT0zTdjP4aT>E#!6v&zL+Myua+ssUL?G6w)DbqB@*8o-?l_^1YLc9'
            'p<B1Jz)!Z$-W~4@;s4Lx9l}~BH+}{$<GC35L39I8k_aX}S_v9)986$cUMQws)CLf@aB-qJcHlt}!UBZ1wMhI)FwQ)fF}z+*0{SWL'
            'y%nx3hhxC=K0WD+Md*zY$X>R;wI#$}M~q?8nV$+zj0u#YnfmZ7YX{N9U-%JDqb1bO@d}>f!u^Q@{hxGjFL-!@5P;w&NHf^BAH+D0'
            'roqh;CYb|4{Cyc`9t=12mSL8{5KlW|5~Sl8_BsV<F1%3?2GBGWcY!Cac~Tt)e<y^oo4nBg0>C?%Yet`+i~AtEh4J7&0HFV25e8$}'
            'RM?mJ9=pM-XC1K!#xq>&#D~s^7HJlY8HdFJriVBWEr0CL24YKinnEjrLjQa?IPMMmhjJOA54)H^`{x~4c$VO-cf`oUZ^Y9n{Dx;V'
            '$~Z{IOPJ0Gpgp55f9QxiFAOFGe1O;zkO)BqQ&}uCk*;9(#~GePK&;-$p-BC?7s&_E|EG=!<C|0-5<tg0z?ejMS?13d8A2zfZSj&U'
            'n8HZV<&PbadJzEkqp$k;NdgCV#AgrB03Kk3EKDTPYoBb1z1<E1HBJdtQh(tkF!PoMG)yw`=Fn{%16hFWL1;>{p?7t)wRIv-Ga%<O'
            '&G2vo<l1obLVu3f=S3?39nX9ONP(E_ujp(hnMoah!2Hrb8@w98ZuDOF1}Eo3ar)}j;9$@@60c8BUjxMfdUbeuaB<u}8TN*Q(-XvP'
            'U!3)OhsS-<c<ZeqZzd-3O|lHVxz})NXw#Sfx78NMbSwskDMH5Q6Mvg!i*$cy2k6%gfv7X_?{*}e@)p6)Bp#<bjPo!=;Sd1f^AY$i'
            '95BSoCet|}fh{2)rTQ;8fn6ewEkJ<6K>mE>PxL67gw`H9BF{dI1JhgLZozi*{M$r~MY>D?)qFhG%?;NW+~8824t$g*dky4x1qg{J'
            'FZ6)SA%%nS02_fCWUGarcD_H5P2&-gq8Ap^Sa>Po=;7dCcrMZykvh%1RVq;kDdd)*WzOUAj0q-C4mfs%bmxa};p{+<5Ds?-fe(X6'
            'Goa9J2uCKoM%wHzfao-EC37!qh<}I%5E=RxPC&4n)6d)B=C&c4a1Po8NC$UdGpMLf!sic8&(8V>guHiYho-1l&j{$vbRQ-N2Obdu'
            '$j5aH{)`vyLj9OAQ^c)@e~H?_hA*RxxAWw52*+;@=WN25iR1U)4R!zw9dSCH!qEmEbhCt>5ikxQ6EKz(0iF_LiGd1wcYqoBEPx0R'
            'o0e1_xXiB2kl-r`ksKH-#k0Q>Cw(~0;=F%yc+TV;1q$_{&IiSR=>T85C<;tm^B^ET5QLds10N1xyO=W(pte)u7(imwb|azyaRlD$'
            '0aFcAAGuuOkG(92qt0fqmpk{Q^CQa^ZJ=`gH39}f8xrw^4*|0liW;8#UIH_`0j2{x0$*pwx#5A4=Pji{Xu*u*@{4d_aGE^0KWQgE'
            'w`3Ogd5a?QfHQ1AxrQU}^B2<P0?!em6f#Q}kL2|WVw`kvkdXoWQlJcKHZVT2q|E=2ZB7~V*Hedh0Dl4YXBvEf)xtoqRdIrXp8SmB'
            'bTkd$qtp9J3m&HQ6)j<K5I58jSS&}dRk!(2ID;4ZmsrGrI)Li%QD7<O8Hidjkr1F0AUK4<9MK&|Yj_Lbhd#4E3D*{JIeY*o3*?Gb'
            'gZiQn1_L&5LL^S40(UFJ(=!hy5s;5eyz|BZ972%=bHA}U@bFZ$d0GIyKC*hGDL1e_+&B=(`$z6s|1}8YXTM%c0g;a+9EO9_lUIY+'
            '7iYbfz(WuFuX-0pLs$e#3?!T`;rt}P&4C5evE3KTMHqXNw51`5KT(cgVFBgI7W|t4F(Kh2e+Z{Mi4ltF*2vH9eLqs?1n4Xhs&q$!'
            'k!Z~%28j#06TmuKjYeaO=$7kFmq>bCSBVN<6ydmtid$Rq*~m-%=g-t%w{Tv=V5I*1k2H?d?>JSz64<JEuK)7YFGh6r7j6$v1Su^3'
            'rKT|<DM>wTQA;@Pna4jr#*yr}@G_i@>h})*+}iq1d`)5GJMjjTBfw^m*j6G1Iyi=%qSKZ7guXDdU_>+(NHP=aHnBYvB>>rBI|!R$'
            'sOd7IgAV;}NEm=OX-rrxreUyHAiBZo5V<^%<yo;x<7EPTWPIyIH>`({Nbw!&4S@YA4w!<V?2}>z#J=;DVX*bxzBnVqK%L9RxZc6R'
            '>BY(LeCOTS>A&?+egKS<XR@Tiz_REE4oK4d{0@MCoI?zV9D@SDO4~Iv8aIkviPJ9ibQ6i7C|tqeTcX?oL<R;-mXQdiC=4S65uCui'
            'vW_@FxZ%v{1fL_N1F4_D9D?En7=`pm(eoy7YZxr_P(kPI#=fv9P=JEz6iz&lHCXH~gTwxrgOi?uTFly23Y?U8N2&P80y<m0qodRJ'
            't^}3)my6S3&pkVp@^x3dLPknZ&Pi{&9L)pT+f+$+uvXG_CFEgBfcK!okZceT>6|R_%#)zzt#Cl-2HF#KyIq2g0e{^E{yh@%$H^k9'
            'dzu#FXqsD?!s9)VH6PW`gmgK21?V0C%0j;p?AE-q_3EtOcZY-inS0oS#RCHw9Q0x7P0~IaQfNV1zeF0B1|P-D%V5I$z!QNhCqW1}'
            '2V*9|42BgzwlaFOLgg&<wBFR<0W(mwU|&E2gQm+Ugnr`2N7*7s=00%UF>tNfG7O=6NQzd1!vbzVkDK@-sO@T~8y`^B$039<YCKy;'
            'vS0I201n~HQ#IbLHwyzGq;mmOZRM?k;*veN<hj2JqE2JW;RER1i{?09tP*5${=`vq0d?4T$0PoV5)BE3%v=~>q)7P!P(Gw>3}e=i'
            '5v(Z1W<aphF=*W==K<-yM~yrML4j2Bxu1DJi#;NCDe5>9pU-;l-Q)h@pyv)xU1Nip`Q><h4v2nn?*7s{y6B(7<4+6+(1PLnbF$bp'
            'TB0#Qni((fflDTr+c<<7;wwg4H}U_zgsqq~T08>C4T$~|YyvyCB+(nY<8<*Mntk}SJR5MFTs#je7~%nDcJQWu+;ah7=X?_Pwzjs$'
            'Ad{vNoqB_JMELut@;A?b48ery4`-j}&}f`_0UpfzTR<64+?KcnZpvK{BSo2Xkoj(H!H$T>37lyES)2fw(qDQ7%SmDPmJ`@KBz4?z'
            '2gD<xx`}&_?GysE%Ms~g)ct^zGa;!Hs5xjcxb{EzW1v$mqpwRu3n(jbr6|sA5n)KY1*i!c{%An7rjdD(mX4?asyY(YjE!=7LZ}zW'
            '1#riySVs?ewfJ9xfUVpJ-Gs(nH(Ga?EPWt@L4qfG>U|77wJKoXaDZ7-N&|U1^`LOlQ#|QFOubt;_dv;`2~eE`hzCfYJ5n@eITpza'
            'kS{NtdowXZPFEN$xDC;6#YASbWNrgYXyv8e3`kN3_$%>?l@~_A$FV1Rzn%=Z8PTeVUmTLS864poqG>*nqE6lhh{ji!gw9F+1mWiw'
            'FI@U8yx%8q3SqlRi~&3xlMIQNB5h(V)Jo@QVw<4k3d5k#*w%tAn{TD<_FG6U{SU~o4&VTe<O;~@#%eC`RFKa@q6r|MBFO_OJY1$^'
            'k-Ue)+#Z7>2AGUC5M+U200J>I3Xqqkakyk82mDGpTj%|Qv;NTS9SwTtCasg_3wT0fUo<7Vdkb0(kS4afxxo9}xBXumjwEg?K#1kY'
            'Ct9XE{>l%r9rXP5^kR7Nvd=8F*dNr{zw^h=PXD6cw$$q}k`Yk9LHt+rCzw$CgQL@n!&gVWv%Wr4gTv~MpwzfoJoBSc$BW?`;0ka1'
            'CwWKjBkCII!uQ9wE^2hxr}wuDXny{#-#>U`Pta<P7V<bM_ds>*l)nMwTyh>4_TaF8G92`dToBY=4USOKXh=5L*isMOliqP3hJcre'
            '-@$DLZqZB{|9;tP|IKTE+-?8yD*x$r+E<@;ThE_9{GSFKu7lH~-b+{R_nZFFySmNRC$hmP2q*)3OXLG@n+2M>H09ZH(6(~}^au%Q'
            'Fav4@Okn#5@v9H?<tF$TnuMw$dv9Z--eg~FYsA_X%{zbP2|vY;OyG@$W!p|m9QrqYvTqKT3xp}D>q@Et(Col!kj;3L1Sf~Y)C(uU'
            '4RQgVM8_OyTbZl{9CKJvu&tVrn0Ij8`xPX;XbIvDoOh6W7I3assYmA#F}`!-XTSlCAsIk0?khCY5Fgl4U+Im<pa;=2lsPpaTz8g='
            '`yw|$gEBvAC0JISJA|k)TM1m$0Aqw89vTCj_!aoT5I!IdQMN+K_wK0%(xbo}gBTo#bAVdD)CVoR9DtC7ij^0h`BN=E^q63w#K)Ed'
            'VB9fq*gK@=iodgxvCNd|LSL$Z$dTU0_mPw*t`mQefT*x@4O%_@%VeJQGL!*G+$18aE>NWHrzK4}zO+TNR6t6D__y=Z6U`g=k}gHn'
            'YphqKI85k!GrqAw^NLhVT4Xd7c*`U3wvucmbhU-{XC%}>_*0M6Xb^U8Zjo>Yk(aC#mjY!k7gy+q>!x9xrLK$WN@E7dkF=NgfQ!)5'
            'fBC74*I7#5oB}n$pDw`{X$FplqMz%+Dac&6nff8<yEx=N8b?|pAC(@6e~<v!64v-F-m#V?572+|tK7FM>gl1AOIbLF4o(3&;6L_@'
            'd_wR#|D&RjnjN&uK>P5U)>FssAtwrL)qS3u@31T1Hs!yCz6?+93je|eBa9fj{~vS#ts+}#Smzh21Z@(k;VgxQn910R51YmZoVpR|'
            'Z9Z7OP5F?3J7|4PKxN~T))A+87_Oxao*4w+7~z3|gS7-XnlyFPpF2ucHs)tcEypA}Pu5&Y>uTf>XrSy(;~3Y>J{s{GWNV0S8u`#!'
            '?2=(}?BZ=w{tf)gTIsnu-7S3#+o?B2!x$@-WZyZ6frx;TCQeQj^+CG}9`vf!ezJvKI(7RwuG+ZYATX%+3=^l&L><U|3jqs_j6=O&'
            '#Q&<MfZfuxPd(_;uY4kmsXUlO04b;>0($p^_zq67cQg0)Q30bXe$mD&@jHN25lYQz<4+c?rxV=$q~Y|y7$ToY$UGEO3H)!W03@5r'
            'zaIF^%ZE8I9{#N88BoV8A=;=GWG<13s-|=x&5gfssl;vwR8oI-r99^{Hn|`$06H~Mv?ux}Nu1(EX~&Q5f&@?zB@1k&2HJoE@qpF|'
            '1DyRuyCQgO!ylua7gZZVZ$;xkxbQ>ftHcrUl5XxrOJHTN9-oX7`Uoi6hD=or0PPzM2X+q*Q`1p8H1fcrB-@!TLl6qG@okf7SW%fK'
            '#jQ$*Hu$Uh>|Xo%Gmy*@P*7oGSlB@IRja7CUIKdx@4b0TY<ITdf81?BqSAJIn~ZgHKMYKr)YuV8$!H6tT%y1(x3p<6Msp+dU+F$k'
            'qY6ft|7T%G7s)j3W3_~I#`8!)UH~elq!*qHh~^+#1vhBh0HFgd9lVPu7|}lB4Hi#K%1Cs5t8M-)Q1R6w;tXuyEnj#MxKSqJCh~4a'
            'cvA$tytknN$@8QmuGRYS;(w7!<g*bjRg+9}l~)Qwq^n9EPOKVDK^5w*k~%;v3}ZH-XY#V*9Plk)tZCiRp!f{dF0mP8Te>e9>(`)W'
            ';|4o8aXm0KjdC2&yYJn3Xv-zBA|;yK#A7sb=*Xe4DD=QM;-?44sXrgn2AWi&xxWHkr+hs2#V0kRe@Px*fLwVuUS{xA&*GuMqZ_=o'
            't7m<wI$u#61?euDcvbV3gOD+xV!UF4M7*1&t^Az@Q#CN{{Hj?1vnR)ax71kC7y_6_CO+=Jj*v_nCM4tv1GccEn)nf|Z{o%WasZDd'
            'I(ied$0z;_=mSrfPbhAkd17&D)s&t((L8aA#$5BMX31*bq790=N?t)7aui@CLG+cbP~De=P4zI4;-KABQaFpo5HXX2d@Hviu_Uk1'
            'qN@ODQOc@uEVV%WSsukEuVu3OWz><b&f*(%t5f@_)G4~trwtk3g1D!Y#WIg}=muT+MT#aX;n$}AkN!9|-@YabA{`h36fzT&7)YP*'
            'u=O#&l43f_x5BBo)_OO5;u@!j!i?w%1CfZT5o)K5RFPOlsh{nH@pzUi%y{WfzMtfv(!fJAMRP<$`to{New_8~Klw5yUV3X&%KXgR'
            'Rzng{aF41E^8SVdOKr<CSprSM(eP|GM{wZ$$d8bEF@?0h?L0XDdmO*y@TsGwC73a2jT0AtBeplyj|y?7Bv<w(6Y}=ZRuje8>4xhA'
            'dTXK29$vC1cf^%%=_=bB>$^VkjOhGMo}71EXG$n3oNB4`9OnU_Mb8Wk0}H;uJ!r0J=FcBGnv*=pr?lb+b!WyJ<;a>nD>eCpeS6DF'
            '(A_rpaZ}q%oo%rrwxt$qJZ}lxa0*j^<1-1IZAscr=!}n*c8gm<NL0G>%4cY3-(}!6%;|kHIvL8B)370^r~L_K@Xmk4LDVev)XM0C'
            'UV?(E0nA%JT)60SmNN+(fa+8ft>2nxwTkvsvURh?I0$c0GEhcNfyAiO_!c)+9O1xz`Y3xhD}3=^pw@XCOmJpwiXpN!TB7kiO_{?v'
            'U;RePyk5>R^_aJ?>WUeibLCD|nU88Qza>}YqD3Z7R?Ob@Ie3X<A)T?jk!3ssc~@cVgMmg;r}1WneuR?1we{6&wC794M+Oni2E`(d'
            'Lm588&T~whq#M77aS&dLCR!k1L0jHRNzWmDo1EV-`X7)4N}m|y{xqDF0NCgCu{-u_M+qo4sG8m)z$+^D0=EPeban*EBb7JgFy#>a'
            'pLZfZyN{EZkkbgFJF>Kw$Us2?&*9rD(5)h;AFdXantKIKzzk<3blbKeV*ZI};>Gs@wD-H^$K?)!B3%JsfwpEO9nAfYPN%4}%b7n$'
            '#EcvJGH!3#1}Eh}_HK|<gjGE2h)<OS$@nx0M-LTKw0BY6DcH7INx^D1-1v$i3(#?35G!9pJ}%-r`8Qxj=zR>(77$AWiF{I45j#xM'
            'X@y|be^wf~l<2MwJN?sT0i@!1w~QS!&%1`r1~4fzEOINdJ8ZtujT2ve8;FH5C|=Zsli-EC)>Jp4=vX!P(8UTJ!RO;Z%$6K$^08B6'
            'N?K1AOMtTf{1)uz*b3_c76Q+%lg0dc=Q^YRO%@XlCa7{7J|{Vvq5!2=;&#dbbm$qVd`Xv4fZ<;h?KVcIjB}6%(R$>ki#Vc?qX>`^'
            '8X5lA((C#jG&9hD*omHEz`heOrS2}d7BA-T1M!62h{E88tU*#O@|IaV_jCl<SLcsC^0Rk`{lB6MS)&-Z(@+i0hi4ZD!wZl{u~{Ar'
            '=5jEYZuk1^^y1`@TS*jQ8=R}733S`x5PdcpS(fTY{o|M1*9!d6uY0dS8phY&4Px8aRlx&FxIzzxar65Y20WrOXA9xm5*hvvfV?H_'
            'P?@ybbXu^Zb08(%2D=hAOarIsHP<BvJ3<IQ?KT?lkwwrggUmLCe=__BcutLdb!EqY8y6OaIO@X0p(Reg@^wb9v%1%?Sn6$Dq%;a^'
            '*YhYmH|N8kftLKkcw_k!?(=<~`OF-JAlBGt$QjSsHvs<Ec*Y_FJ6pobhCGw5u{Pye!oBbw-i{W~z9Dp0L~mbUpi}$rOAiD5q?mf='
            'O92Q)`Nxil^h~vbbYXUkZn;U6H?-R{PE|{kPLO`24vPURBnLkd!ginhxa;oj?h@j5pX}ysN;s_JyjmRQ)zVR3Eg9q`zo5O9)KY%@'
            '@4gh?;+JxZK?+>XeKS^-q{K}qVI^EBR_dDpXRiu6Kkj~;o!<KzEBdcr0PpX?zP!n4tFSSl!^Tv&|C7#-M#LJC5l2+-?~{K*yk83p'
            '{7<r-x)43O0FZ&g(VUI57_(OqlGkZe;>eS)gY`um+5J|aC<wlX7k}#?4UhZ#h#K>t6un51j^s=yem#BLLcieE(uh3J11S`9jmX3d'
            '-IF~uLZY-Bn4)n7x*qNelx>;|_M}IX4&FVEgIN}cf_J?mHlauA^n$F^mZNU4LBsQwLlRK;DR07_u?Ks`9#~=y`f%6CHZH|IG)hRR'
            '7D|wQAs32E{^;dVFnY<4;-4H2SNlI^m5dpHoq5#gmr3##%!~lo$00?+wSv{&t~{7eO=!{2d;o-wEr~yJ2z9(IXjkXM{@bJe$#MU0'
            'k_s8|vY>{>055^TV+lIostDY|S)BV>gwDC!IPkvcbUNZ6_`lm|3!4S25FI2w7M<}c08go4L^EE%d6$9OF$dD-!BCFa4DfTW{jB|T'
            'Z`2O54XLEUPYLq9=LX21xzF&op@hu6GVUoo(@n>>{@k<dM=!{aQ$aAGbSkh}9RA>~5(-&h7?lnlt>ww{QQLd=-+n#3)2~3I0NwB='
            '@$OLCx0I5Wh!>bPhFpW{w_Ccz=c8zfwEJWG$?lWqkAsOcRRyQ#3QoHgWin2hf|l{kMyrBg6~Ty?3#$~koA)1{MC{-N9EnAMrV6G|'
            'H(r9n!WkpJcjqOkw+cW?5N~_$dPjpwDE`e);x-$D>6`?ql~ib7np@DbMF*pxg(Xk6A_v3&`hPt*x<I2qQ@JAhAI9O5LTNGfR^<h9'
            'TG}UPiQ!wKT`#}EM828yCbvs&)NTptAUfzELKR!IOOZ$~M9lV@83_$5PxQ`@dg2DdAV1ddJ>>2%%4hKPI>BrG<SW?U7p&*er7#Nz'
            '8UM!rJZkb7)PZ9xqio5+s372dy;&3Z|2xe%7nT0mXZ=7yVZ&R$BHvY~E{%^&Fa@Mxa5C(lo%atgp%q(B_lX$25S|q4*=LY!u`2n~'
            'h!W*^+WJoMlG?(Gce)*EF``(;qrv%5yV2X0By}-x>!ofKM<0PG@|_kZG;#yu(l&Q<$E|<??V;$(R`s%Od9Zoua)5(*O^0-7by>6z'
            'UWq@$R#2Lc^6Lq|mP-gySKE>~^3+wf=CzBVNEQY)2F7Dbr7FWTJKLfI9f4i%F|Y^B40bCM%pj5x7}um;(~cB!<!W|#8CWp%;mYgx'
            '=3Cq~3^BzEtvD6UckpdrJasza6rDU(7P~2|9vwW{^%GGwT~f%@bQy|ikfa&K_+p6X@bukzp1^>!$jm+Dt6~u>@FrimS53Vr#e?zH'
            'ZvcP=4*PBoeNB49!7r31pls`GRTvFhbq2FmNlbR+XI{E5FOJ^YpQz|sVX|sfimMJl0|m17?oaK#pYXam1XUsk<7~Mvio{Z;Tg>T<'
            '4E23|dUkPg@$2as3<jDUjB=9%;@R#qj6z<n;sh_*8&NFKFBh?2d%k8d?9M%f(WM>nF7y!}mJ4+Gfw5Nc5`9%B)DGH6#bc)gTufOo'
            '<F_z<yh=uL3c@NsgQ5I6`fjdR{G~`h#C=bADNJ}3hz$M#&Mx9`vP^Kwq+l(hWfCrD(4UqhGcSZdt|vGkg)@g8@lsnta_btUK=i~4'
            'h&_C`Be(Csyx-XnOVrjM%KM(?daFk35qp3duCMQknpMhhOSOBYFO8aOC3@(vHSC?gb*Y;XOne(>^O({kX!)#oWF&Lu_#2h;iz8@j'
            'nqs6PDK+EOqr=L33$jXN_02}haORS`3#MbTzt6``HoOZG56B}DasWcMrlrb7QMDwiVV=?gX5f3#a#7u`WLneBa?DaglI_Pp9r-*r'
            'BIwr=6<Q^=t^65@4N5L~=O^tZnig7o(j0A#6l9I;h~5ZuaER`o(YL@hRU*0P1RaDo!#9JIw}X?{?ot02OnkErfj(7TCK(3Liu)P;'
            'wm$r#_WFV>st;?h^xMp5!wFTMg>N*Y4X0E*a^GM!n@yzpP=BiLY+SF@UH#}b59<lSSN=*hG5N|LsU$>S`5RS4tQtY&MMf2N$W4~&'
            '5ek(cb)z7dXQJglF=jkc0yyTQ)v26iETkkqXZ^oi49@z8E^B;i$&aZN))R$ttICsFZdN=!<rdXwU2a!{ndOGn$W?Avi67-=Ra;yz'
            '8Ch(~xQeLYQkMCBEt9g0hXv$NzUHhPw<!^b82cE6a3gb<QXV^yWqa>?zlxVQPfqOUVZv>hw=KP;RH)dSGe9GjMHo05lKc2R+WVwU'
            'oN^Z_hA8uaBhy<br&U(RG3CJA2kF}TAl-pDO-XvqdLX)9%ESU0rHlj|=WLy%{^ES*apbQAYFPWIL1rMjjwvb+M<;9WJeagOfkBJ%'
            'i>J!D9=4iO9E{_Xk~yCD4^B@GDMy|E_oa_^21=<WgBX-GAWtW0r4%FtikFxwkfH%NM>Qv@!<&lCOk=04*fJhPWdOy5HUTC5wuK4G'
            '3YDZPTU?q2VMq?gRzC1D9J?}|I3)~A<$*u${nZ7T=H11RQ@@-WZZDMHNc4*CPF?0n`!b{m1Z9jE#@L|+YMjmg-1~8NR}9b4Yhscr'
            '=i2WC1sefup3WrVIK0Mm2?|-V+My(Ms8p$JJ{5@5sij7H&)ug#JzGPFtOk`d$ZAm}KjupXnwMqJFqOvxk|)n3^TfhiQR+}Kadg%Z'
            'u%Fz$KR#Jcy38ikM9XYZNv@0rYB>k{x#|WzDB&Qcl;*{7npi+B&Na$jib14P&2{RDp8VE>G$NHS6GP6Md^e{|N0^89A9f}+v}zdk'
            '9!a!h*dw|d3zorUatTiErjHZaj-ptFxf2a+0;M&s6M>izDY=PtWL#|m=Aw0}`Ye&P4!@2zBo&Q=tz>+PG#?I!{o{9})jpFDva!kr'
            'PfL^yZN+>R6!?b|<3nUX%^tZ$R!(JfBMXu$Ymj0pannN|uZNV-pi+YaEot(dD1$dYuKY<%T?ZX<aIwnM6;-;9hE@xseCsWpG)BqP'
            'sbk7Eu$yuqvAKUz6`QIbY$>H)C5OkBXgV^Cmy?)${YaQxdQ&AMu+GHXi-IZg4rwxc^7R8<awI(&yy~A1U5pmPL<NQ?ZZ2E8YDOJQ'
            '57#L79uAM(H>Ve8h}%z|b(9s^od$k5X#yi|QH*;_9mztt%V99i+HC5i9>8tl*>XWg9`n~TCr?#sq98E?Oj8*qf`f=xgR}FYvg8Ac'
            '2ci{>_pRLfI-?w-@-@5kr`J2z9T4E>^ke3)(rXR_&9WpIq4}01lIt5J%~{ga_tBK%Ky33M+s-xcLQEEuVl-Vw1yl&_*qA*+iivde'
            'HlC=Q-l{f)CQstdN7wgKYzn}j!quOLmrzwrjSv<BJ%#C)I>9TQZbNU3V@E^qYRmhlLm*fb1kuOkLr3wHiFaR$>?;~m6G^r>+g}*W'
            'e6d|)@Z8o^h4LhH6<Wb(KwZLT4NgLU8v~!V?m;pc(cDQvUV=lu({L`^dsjMP2d@KX<!VE%idNwH*l``-Zy4HPXXAkd9Q003PX-6Q'
            'BXMwg3<u|MpYsuuWLtBy#aC1;ROXe4er9}Eg#XiaJj2jJ?z_D&E`v#W1rI18#kTt4VqnhpLkG@@qBLHJ>=P%z(UO=ro>CU8+cGBh'
            'd;>%G`FzFSZtDxP`&+4alB`N9rwp-@wQT4fj<eaTOCHfIq*^M|86Ep|0H<s@C)r=*au3gAzigpLfnKM?#W~r>$O>}S`PRBMkvzqn'
            'HaD4(Oh!T1gxX+mIkYGpyibjI2Evkz4`}cj;K3zqT7&kgK^(scogMYam06aIWH7GE3-wT5oMI=mJQ+>?srt0eIiti}C(XpjI18{3'
            'PR$b7Eg)Y({#9cod_L?|;IoIVszCBupxK^YwGgzMX=uuDZH0359N}y2X;o)6V=29_N+Z|_q$(mo%h>%z6lJZQ()cgm14+N+aG)cj'
            'R2y+lBNrJc$6=^k(3^gsFo=ti9Qxswb+J~p=HqPor6|7NoS2R{WY$h;Xqd;e2&1G0MFm+2F{#yc;ky5tzMsSk@iF&vl5T<RKg&Ea'
            '^98C_R2{@m+L>|jOSBTbowQiJOp(p#WR2yWA%~l590^r!q-w{5lff~pH=Tnf#YvyN8;okt#OcY=uR3CPkp!q@O9BqU1y*oV#Z0Is'
            'QF-%>uwb13{cRlbV6b7pBW_fh#~hgBTYo&$;l6l2skDf<+D<V6v#f7|G8zr5W`<GR7{o?(PI#|C*0O0dY&u{h{x7stWhOfCcT<&@'
            '=q$6bRuoF;UKWya(Ym_{qkkkMaZ}{Aj!b@(ODqIVUiDN>8e9F%obYW<;Y;v(s5_1u5b&)Mt09J=S8=Ikc;(2fD!HFWS6t4kriGh2'
            'FFS8sDQ*!i(dYnp=Wafl%u2O(#M~pKFqrunshL(dkvKdYpX~J`RQHw^n!(gS0#1=H?omjS)sXI}rW_+MokcdOYL(GATXIAkglbrU'
            '?t#80oIP7wDzzP8n~PBDfKsdx0;e8H7^(SQB!^=&<jCiQvpb+yMVNIho=!QZD~RkjH=KfJ8DSVP2<tM^UN6j5&jC;r3TR*qApMnP'
            '1}OK+&r2n^i_Ti>LCzTJjxd)Hp1KE}EXVXDQYp(oUov*n5&u9yHF+A2nYgveIB}^)ywdH=ujj9|Z8lG$sWMr*c~~fCv?Nb1_3)}F'
            '!;~UKs~yAYdu7$Vvh}?<p0wJ_tkA0v!O4BBhs7S{z*GYg#g`?B=onRQFaInFj&{5%x3wM?d(^qhLGMkR>59|LNf-=ZVxl2q+dWj+'
            'EYFqyx+-Nil`@Hi=xmE)$Gkw=IN7g-Z6Zy}c);;0%C=XyNW8?cuFRakl$ir0h*?0T2)d-^T;@%RCyl02%n?PYM^H~=;3;=-m)~e0'
            'w)FH`k~Fj`$cJMM9!%BvCBdP%iVjv&h$aPVrFNLMIJxf2>sDPqF-J7^lq#s;_b~vQewgYO?G^*IOcWRz^4cm-a!Zek_{41)D4ai+'
            'H>N;lc(&Mpe-%fO;FVOUv7ZCHob-2z*ogGp%2&lHjiXr3g{6>22~RacGgsZ<@DJR{ls|75i7^7hRsC3;0!w+uH;mKedKrb9-*<vE'
            'R$0=`DEKUNZ-EZsTI$W76+`FvW+=w_!^$2?;xgDFKf0=Kt|YQzAAW9j%tcR3v2SEGDP)r_WwxwMgHtC8(;k0PiJ{G!3#~jKbYmG|'
            'IqIlkN}vtRT@^MLxVclki&Z8FGy6cb*a+?GURCwZ#Rs!v{?zDWBDW!+OF_YCs9TtlfQf$?BFAI^PaFNPb2ym#oRZefED}|@l@R;@'
            'DXKwao(nYcx#(NX)fl)^7+PAYDufyR^5bq7jw-frHK{8^0<QebkI<`C9{WJ7U=KsXX!rxgDTP?{H3>otS+1((Bg2*EAr`YhpHwVC'
            'mb{RzzJ**XRCsLU-GtC%j0J$oQGt>gA(N=EpX6ekoc3jDLy&A^S7i;dU)xaFO#iIbCT;qw=F?R_T7u0qmHL0HI&+WetJ58A%6uhv'
            'Sj&T}ZZlC=Q<6ZX%2v*wukiy&S1PkvZz6jW*SBV?zE}{bE0O>zTwTR8(fm{xM&<vj*A$iTP+m+Wg71oEF>Hq8%0K2`DngCnUjBe('
            ')#Z3UuA{Y*T)IL-V}3pJQ+zEG26!0NZrzK^3sS!IXS8ua>;-y~-Zs?}nGsyu^#CaqcijiQG4_-zfJ_})h%J#I)slfsjI0fNbct(o'
            'c4tkb4162oc+?E&Vm$vN&&0(Lu)mT^1+6twf%Ax6UR$@&IVu-Fa$;gH=E~VxpL(^lM9V(b$vy_rG_JRg@tpGZiBFaOn%u8jHotw2'
            'sZJF|z}9W!D~|Y6H?JYvB%pQ)7n%TMM_<sz-oyo=hE3~LHg&+XWtgQv2NFd=*BjWkKHqr9<H0tKQS2t<{xTc-2G;oelx#ZY`or`E'
            '>vgS?HRYpeo8-MQ)|<}Hdg4?4*K4^kJCAr3T+7tCfi5|Ows=%Whey%?o%mQrHlD)INL5`L`(c=S2PRk~I?d#jBpI*u<?&`^ZMN#c'
            'fLN}s#7&vls3)|hsLxi;U(LL`2HHAer#3}#t-pp&S8UnLEEFZnA7$)Cij0eN*5rL2^FDT|5@3g1*_zZDohCTjqaTf9bXDj!mf5uZ'
            'Q^Rp8MsRa@H=F#3X{VuvNCzGE0n%V}9>G-T^B9~B<-S>*PS>pJQ2D7c2jemLOv3*gjH|1EndUzm6d)Diza$GbZiB2`;YH<mq~;qY'
            '*(Og=@tqjQNwQpI+!>E$;uDoDs?m?$N^X5!Ikf6ptKwM>RO5sg@^lV4yzXYuQB`9X5fu0$u+A}IXiIlnC&sQ3#=p(Atd_;4$Wwn^'
            '>XfPKG5eY7OcZ?mM>lK5>%TQuf9lRt`a=S3J$>~`G_TJtPLzw{!Rg6KUwO8l>jJ^olv9^{zM0@U9nPw4Ug^nuycbm3cEyEI$n#s>'
            '!`e4|O!92QPU1zmKS4(3EZ267FdYsg(Jb8flg$%#P+bHk^~V8+NCpx7k};FMbRY94oZ>WF&at>NG7C<>jegG^oiFg8$#I0ir2HQ0'
            ';Z!4x*ns;|XA{eAO!$E)y>A;WiR_!Ax+e3cMI-kBCGT0Qon^6f%rXMfQe9=PRo^!6gG_sN?s|&8oxvl4?#`9@SrRa+tkVe7OAXtr'
            '+}4X<!n~REL2db!_o?ePsxBN&ZCImF9i5LnYH<%v<>c^IO&9$xcynYywDj#8Dbyvqyx7a#tF7W_HsW2?+r>-Lg${w}mVAwhd9|7)'
            's8GXhZf}qcG1nGiW*Llp_0;{;D0!lzv-*SG!P#(E%crV(n+*g1nv48AwNV6UF<)dWl^)gAZ%F7;e>qOkx5^sxp}s~(iL7jdAYUI4'
            'b;RlSIBR{G!<GRsUC9nt`3+pXx&E!~2_=3|Yn6o%*xmaNU*Fxke|C0y=AQM2efMZ^JQ%uX?~YNVqVq<!P`U@MYIj&_H&N|g4vvn{'
            't)Hs8V>2K|7^;DaI@_Do-7gC`vK!W)rmW1%>{dNHIR%vlL(oXjW7m}!7%0{6k!{ZVXTS8%D4{fEaFN#~^6hYN+y@#@DWZe9A7hqj'
            '*{}DiKm3*Yc~hT4=%&C{LiK)d@=Nb%a417Muw4+5is)7(uI4U=-f=uQKL;RSw+e14+h3f#Jvn`kVV3&VgUg@e3Z#b^S5(MiEc7Nr'
            'aFX#d$ixUVubH20r&O`nA1^T-aa*}{P&X`ZK0NE4oDX1DE`Rgw;N*}pD45V|2@6_|ISVdn2U_^}Hcv8wyamfjNq3P5!*6JRI$N)L'
            'M@KJv2XBktqw`dSajmMShw7<DftE2q<*e18_QtigU^S}j$<g3#-+hM>H)kjLq^wHfy}?KCa<~0QE+X#!*1r0*x7&J9?ul00=Ffri'
            '+XyQHTw=qYLE8NXqHmnYv-^*c?<Icnj}L7mH*ILxhBnaWxAgnV{q7a~^V|6kwPQCgfBx62=Fj^y?jmsc1C#}Z8J5#^F8}>se@lOB'
            'Y+wDbW{T;oZB#`!*W~B|)aM$jPYJ2Ha7A9MDv!%J!%6}Z4zTia-Yt!zbVexLvXxt4sK6zAVbWbKg=xv)qI4ByXeA$$6C`)XxTN#`'
            'aSsmD0pQcGpxeb*Er(cP?rlIG|1FGWZO79#h`~292ne)a5T>vcNC!A<aC3vfY^nkZ6-fd07?nKA;Ee%KGLa(%p)w{@HE9JLA`xdy'
            'As8|hWnW~J+FX_rn92g8f(eX_u1i5x>LbO<)V$QCw62&UKTU;xYWY<$l;Rl6N?LWZo3fBL@feJ3h;AG_v6v_kjxdH(q?mh&8z6&}'
            '8sYmmnd#J)vikN8U)AkUDw=eefD*&;xIl^&|H8CcadxXKtmFkSWsr^$o<nxL7!!0+F%DG|3wo7eT#QR+V)y0EC>!)m8ecn36~@;M'
            'WmA>9A!}5(x<ShfV@vuCPh^obGAq*14Sd*;rOQmCBfph5sb(YAQIRwQgAolbGL<{wV7Xc^$=C}Ne^IqfOA4|-Au&M>->RHJ{#L7x'
            '`8^_n48LU9bi(*v-Z9`zJM?GM(f6wa37prn-MyB0w)>Qp-dG^|rgYy3kPfI$cBIF`7Rv}JF%@{g+JDyj-?j>dLo6bIwu+p9mBy20'
            'UtB2!nD1*g7X=z?8Qm`BgTm2rDW>?HdoLA6jl#9FmJ~OyXZ`r3G&?QL?8zT#Ow_;7zs~hP+<MD2s{k!DePm+n4N>fe;v!$MS5FjX'
            '<6?+k>SExRtHegWAeF71$~u7+f@j^R6A+MAn~!M=C0mcTIWA!?=CT(*?mof)d5Zt@jB%L43nHMDEU>PJD4?@)PcOk0uguUQU$Is2'
            '#SM^x#VIv$EhrZUqx(2f^=9IG6ZG>%Hvr$CY=ntYYSUk3CRrMK#ML}Y&w**)2BRP|(n_y~Ah3HGyuJxUa>ijjmlC`3eOQlIRhbY~'
            '<Y%;M&b?%Y`Tp`rbD44Vv}w4l`FN9x*?jS*{3Sj7Qg=a)IV;W*!`2e@PPI0?qi~+`6k>;E6fS2p+(P;M=<C%-Ph)MaM{2cW>3M>E'
            'sa2$S0<`5GsMhVu%mV47gE`$7t8(1%Bo~k*2TI|G7qNbH?*$}0cmm`ZwDO_UgQ8i8**+pn72+sKlhc?aKA`am<w7js!g9wPtstoV'
            'jy)3Es5~bMhm-~og(j8~m569Y#dGvAxrkr_5%Q&n<@(ZBU#4-C+L3MX{vuq!q|5kGxg!)$#XnK5!>lIeBPA=FIN(R*A8!n>@IdIM'
            '?3sCcNycS)R<YYC7T94dGTw`A>7q@K8vBji#&+IGy<~$ysforq5D-$lr384pq<mu4+86|9mNTkj#oxpcNtrT@Yg?ks1mj_oRl7$@'
            '><;hZp&xoHR$r(hYowyTju723X+blPCD$Dkpx+FK@5Hq+(6zV)f`@gzJiN%piB51#tP-X_PzV(#X9$clhtHwypW-fpi1R7nVOIUn'
            'j{9^kZHQ^&-OQ=}TcqPQ<N@gDFkd^>h0I}_dhGObi&=dvl;e4)hHPoea7vlWrMQOdQZIQH8dQkJvPCK6oNBy-)P)G%FyPD74gd7i'
            'pP;>;{o3xm0?oZq;MediIf(pD`NFkia4r!2x%Z*D+rfnM<YFSnxrAY_obtW1w4gS1dB>J0smaCvs3NGYw+z7JMH|SY7tL5;6_dO#'
            'Dg*e-gtxi^ftK<Dgp>W`dtmQbPAKFy7j`E9EVS>))^tsRgH(O9T%nMC<mu$%o!B_`QLI#FS%>5sL#fua!<ixc+*aNTqjdrA4mzB4'
            'rkO~{{!F0#t?{#AaB2M9{B80>^XL8FI`FUavy=YZFtimMWuUQ`hAw;i5t`7u4r=v_=AL62j8*Cd$ww>JzF%=pD0KnQo=hU5Q`Q>`'
            'n2IAgbt55fHl0y5kmx7C=-&Yx1z5n76oK`B03_BHm0__FJaPokNirxugM3~j8f!72;@YSRCUOj?bmg(pfi32fDSI+*bP*x1_3?%&'
            '#M<qc(5MTt|8B`{GbB0=LO^c0*&(GJx}dFwvM-rKw<+fW+jBZ9>B9sIN{XiE$jS(qOv}joBWufZ@-j>tf?UH(Np%b{?m&CJG)Ktu'
            'R>V74!kVMY^-3I_p1wWjlpG!s+#So(DQWLcK27Pda&dM9+zMz!ZkN(|TN0I*I?dvQvQDk;F`?QE&PhbsEp!_9Le2&@YrvpYW*Cx6'
            '$w714_=J^Y=xBFb3Ve0lhkfx09z0Z8nCinCI6vI_YPxb-WXe#ElEp%?fc@Wp|NUA2u!kAx;b%j|{MOHe7G!^a(~s7Dj;%f!>Y&Jr'
            'Doy02Z^#avbUA8<-h4Cxv3p-M=lax(AIhI2+QBW-rreieqqUda`0;9M7Ok0cxYz_3UtI3X<9GE9ZtmWBW4y6zqq2g!8K?0<M;okD'
            'v_!!TrO$VVtkfW2q|oe7F#or_)qDQ+Z}o`dEO>fuJe|VV*%hrRljbe6@xFG<Rq56=J-RqoYZDj41Hl9Ai}oMHWQin49G(wbqWzPI'
            'G5O(rkSbm60{uGCy_funStk7`DCY`hU{U6a79#+Ot7tK#vd;+>0Edw|BP5q}^TiXu_;Um+iFe0TM{RF+e|MLrCNH1%(#&Xd65nCc'
            'q(togQMPlKat!8+(3jhi7nqTCX9TImZK;2vnHr{STgmaJT)YpWNqnEmyiFgp5&Js(hzq@@{7snjX0j&543(zVfRe?j{|0{#;BVcE'
            '0T0NY?^?%8l?!!KwQn|hK%j%Mw{sMyZZEp=LqBae4*FR05KW7T%VBFpEO~waBbwZ_lL6L-T6S^7&W?Ec92nUnZs%I!U4ckuF9V-Q'
            '!7U+~1d*|cPg~+?OYE7VgPZc!c4gxq47YH2o_3qhMVsGs?tOm-&(MeG|B63A(?2MWbcqwzm-hDbrzpu-{w3YM`eW*K;d?yP1P98{'
            'nx1>|H?5^~`1AI&c^sjuqWuv5e+^y4qKH_ArOpo@XMz?9V;<%@<ut5Eau(LJb(8qx`WZ-E0@8sY2O2PcGLb?2Nl%pMLi;Ce24D&X'
            '-$%oHCGo28KDR_GIM~}VJr<=QL2VvMvb-G^BOeo(;x%>)v#Ky9(A1kzR2q@miYO9|(C*{LLb}43<NDZ#T)oH-MO_?*^#!B-qj4B@'
            '9StH<?i9vjFSOcQhk%`%ALA}IE2bekbnqNAoL25L`VQy;3AmeXwR)AaE?^2QK4mI_9(yJf_fWIt{|@`88Ns7q-+Tvo1W6rn1E`m)'
            '0A)rYjpO+!09-M*u}m2EoX^UDqeln!+9A^9JWfZdCfcrD8e><TPdIXGLLw_`YLA+5r+ttE<_kd~0fFHhFbNV$#fT|6<5X>*N`{np'
            'sq)oF$F4XUFOvj~co?HH_9Ch9guX`!^=`dz+Q!diek3Ks9UcFoLsX2w5d58oi*y!VLJ0zAFsmm2k;<}c{s)YIjA!(xl%8QdUG9j2'
            '3sCs!H2A=A9Q+&BG;MUC-KUt$J&Rz{bRHVIdxzmku7?x}GFi?+s7GkR7o!xkA=gXCL7?i~V{M!$>pmgH;5SVERSFYfV5Rv%m@is_'
            '*Qwku+*hy*NAY-8^neF|$0^o*iP0%W&<(>~;E7W%Z1Rs&7t02=IDwak0%z#Asg=CslCtaTI0pU`QPy9ThMTuz8<b?2Ep(%$CvD8*'
            'rQ?j2XpJSs3~Pk5yc10L))_6kAT&ymks2KZ(J~hr0f-6uV&J_2Kqc@E9am*47Oo2`&0H5`h;YhqX{pI~_sHzAh{Lc8zkY1-0@-Tb'
            'eGZSz+%wR>W1<o}005YAYdQ!N!=IX3%g?lAU-*ptX?#X>g>EKW<S*^v-04J;Z*zH63{0-in6A1~l;cYxfxeN(%*u=V%o9OjBNSkU'
            '&umG!(r9cf1W*<$!4e$x_koPTbBTVyv*-`RLG}Cn8AfI}^=8d*x>aFp#skpD`KM&G`V12{2Cy0cY#Xpb{R?<90A)@r_Rpo=-LEC;'
            '`6$Z>_G#9XzKv!0R=8i=zU76bo}Sk%(b6i`Eb-4>jXl+ae8<5m@<x(^WK^9r%;k@-NZZ5LF9Z=6e^N6Nte!5YMU#@`A&tPa{Qw<`'
            'b)mFa%z309y6MK#rsAs7(DxTjeFPor@O&_6!}kC{7g?xd`BZ3>t=0JeF~?z?`c2i;*{Z>LZ5I7J6tF^&^EA^({lnM&v#KQ1d{ao&'
            'RVJ54kJJO(^;KEfQ`1HIU~qPDF&Mfp;Q+pc!G~|o`sZ&>kI;qqY3c33DZDv-e}dL8Xt4LA_+C8U)qLkKIE^FX1Df-^!z?pcI{*k?'
            '|9m(&?qOX?<O+Nr%vK(mU?Vrt=Cw7BPBBBJ>Z+>Mdgw(L%!M~X)*x-G=&x|6uk}=#T~@s5dYSXO;kcnU!XT!JxW+sgSXg4h!3EbH'
            'G02$qrEKlM3r-MzC~h{QyK{9rUKKnGCjv`$-1B97s`F%10S~^%M4PeU8mzLyu?HlYY33eTjVV7ReV_O%394$gdR)piOrS!`2q09R'
            '6Z4RK7o_r*!slSPn1Gkd_{w<I;_C|KS7_MeSn%J7(@HBo$o=j#stfF^i;V8Bi$`6<mzgJ%ryM-Y6V2PlQAgRs%uv*n=Ggs5wG}-?'
            'MJV(7An5X=!iZ$+>&gj|tFCQ(%9l#jMm71U2td|jqGOZt2Sh_Do-|pz!~asdX};w=NfJHmAj|q3!y6}?Vpj`w7%M}4D3t>qd&)>v'
            '4sB}u+CS?biDB=}@!+I`R?g)Z{y!2cKbv6;Bjp0YYK`+H7h;^`X>H2!#OS#vF!fD_iEywG`e&rRG1A}I&P1%Vy9(9Nf1$!82}*V0'
            'ggT94uDs7lN>s>Qr7)6{m{lQnp(_{Il!?B|Cp*u;^y#7a#*3CPEUmg2Z2FLs6GB{#Sq+1gG}c^cvLBxvXP+OGjf3rZ7&F+OpA=*5'
            '-OZB})c<E+Tk(gaM?p@$$`SareULaP__Irt$2DoHWw7E}yLA}g)QIzvZ1FhtO**jOQsQV9kN{H`IvWP)*HTI|YF}Owkv$Vlrc}5`'
            'i3((7*s78rHS%FStQ`4liBDCGR#;VK|1Z<^OY_8B>A545v8{SqtwNPsEAEDdWPh)0pwTT``PT~!pshkIIIHcKGXNfeD(o=VFBjLb'
            'dBWr6Whi*r*c}qkx+WlXfNZ(OjZQ+P#JZ{r8aZM`q~&^UqjNQq*Qhj1x$HIx9d;{P^Go-hY#405uOyfD*Flgnsyl7;q0+P_<Ox^R'
            'R9*<w&m%9C?UFZU%{ZJ!d2JI=0y1h)$0qVtz*^Tf8pm-6FpOOCO|5UoELA(-Ki1rc{d1|%m&`CgQGj*Jk|Bhy&bdh1-=tBuHW^q+'
            '>X@qvX{=d-<kopwzlwn5YMPiI4lC_3<9y*zRTQ%C8g-irFP(M0vvs|*$9hx#;`(n-Dtg<QjIF-bD^4SX_gAhlrsP6JbDfI{kvywu'
            'lRs7dbbd+`kYfW>0QCyclgiqd-Ii$9t{AiqbZ62dSDxXkt#Ty2j;f|z&82Z`{!EYMD#~1p^&f)?<(5Ob!!2GPLYoQ_E3?B%isFcW'
            '!HbVo6~VpKt*+>HWL4Rw8U)Z`w`Rzy!+upVw8;BpbIw(i*;cmj(Iys6Y_~4)95ryNjBc`P?Uh&cXH>68FTRQbBPWTI-JOzO!k@e1'
            '`EH$?FT=YEXjiO8SIRIOZL#EQ<z22ndF%LNd7Sm~<kwi=m#yvhtgD<B`*YV|2!x5>o`!rK_<Jn^WD+Kl$Z1R~q{5b{Ua)fV+^x)F'
            'D$PyrG2Vp@VaFHeLn_}e^3k6gwE(nE(K;xbn5<+ejbHfRX{3wCr4-3R8OpW$f5D;#>f0~<b6to_7G}w*3W`Z+xJH7hX%Oa5VM%32'
            'Wg5U+j+P3qkrjN&`e#_Q)m>#-<~2m@vaDNA#qkzShIf-`ll<kIH55|^GO<A<Tf%-!f=5dVHH%UidB)!*n71ko#(@AHGHVnwxN&fi'
            'MU82qB@LGve#T7=-`Lj^Vj9n>yX4{C8_{CxW3u08QDdz}-itrU0qVb`Gvt1vjeHQzQIG|2Tt3l)RBGJMqj;sepdw2z$e`6;-kJ{='
            'S{u2I7Lx+wEy{Kz23gZskgeh|nE==ow5)cgZ*ZFH%)5nCw=X1abGX$c4wsZ*m`2~O8h!ifVdoPJ{(z1y26#aZUbmQe<znXR7o({='
            't!KW`dM4g1n91<5;L!!~7v!4exTa%cP1(9NWtD5n<eI9<IKBGqcQd;m>W2R_e*bg8#6{XIVEyN(3?VVJAYnx+XhTrsEz;XK+t9r6'
            '8Kk?~I$p}|^jYTcdgH{S_*eI#M{E(Fxvh`Qm#r4-Pk!x8ZGCM2agFV<sy=mk@?&*<>NMoXD*M!Fip72f^MmeYJuiQ)U_i;gX1%y7'
            '-BBoWQAv?2+3i`JjMMCQ%9BO=B3VwVWy$YB&YN*=x0c9Ll~p8Ht`TM)5k_0WHruRAPg;n%mA9Yrw%yHcG3hP9x?6k4UNf#5LGKg}'
            'zV0gi_fe*Lq4IGq_Cr_fZGAD#XA8=1i`?gH=s#IaHkcAi36=A)%#!Q$yMH2^56Z(9B35gEYwV4@P*up#e%ZADbhfOJhgtQvN_OG`'
            'xC`dKb#~UWf~@7y5?#X^ss=iKx~Nz+>b3b&KW(V3nW9Yxi&mr>%bPibWfCty0~VB%C(_0?Pv+*V*=%VzUx&u_rvZFb->^UNojBy$'
            'h~zy*+N4Z^k4Y7TkxoVGbM;Oln;Y#aT-nj=xWSq*A8l2lzWpyl#u6Hv*!Wmy5adfF|Fe*k(7L`u8BNKuXS$S2qh7a)aX8iRr0B^i'
            'EYTT{YOK0N)la9I6r*Lh<f0~+e(IU?0-m5H1>?cs>l06?e7ke+W8*RFHo8~23R?R3*UII#_WoINdF{sJL}zPL;-RN+UU*kcTFNRd'
            'X_S)E4ZDVnR>~u)(WqeUl)Mf{m-$~n2BYld?6e)}XgxVuI-U3p!!V}3>mtMRPAB<`Bkk19&UcC*l53pubZc2225c;oD^OWQ(qJO1'
            '-LvBx%n57T%X|3|(2|sEoO%@nBiiY`w;(@pj}ie^8_!fmX*TW2Vm5NU81ubomeMo?sc*!yS9lTO-T^+7Z4C4DB_Uw70FXxF1~ksu'
            '*cTt;1dR-nWg>Kb?rmSpywwB~q6djnbOp*!9wsy?mmw@_f|P`*80cFpa;TPAYQ<<o?f9p%j#jp$i`W_mg^K!^R(5!HxndnWP~b{l'
            'h?JbK2#6`ZnNW#bAO!x$j#a2oTP8eooJ=Y0xW3CY%QBH~3>PUa-+$VcP#%l3s?(_ZkKLbk7V)CVjf;swHuh*yIh|>TOPW7DRpk?y'
            'B1!LER(IHkW#4q7-BkN|q=v#*RC&;h$PR_N&<?pSZ&X{J(gjhj6*k}N;tQ06K~2(%`|jk)pCm<>&n-={ly)7#)Qx8rjPjrP+NvSb'
            't$tjBll`Yzc>}#sC7ATL<iyF<sE{q2IUk4jt(Dj{oUOYOfYCV<+{EY1hSCT|eOg3d#Uk{m?k)6f=2hy?58ulC5>CfRA<A(&C8KyT'
            '*@x}PCcjoljl(zduyOQy%Se@p?3rGo9kQZ~J2#PO6K4L(&SkZ)Go;Axnk~pVEQ<f`?Th{&p6<z@Wk00MDpVr}(2uv(CH99`8DlIO'
            'hj$5S@eN;{S`W91N!xER7%EH4gHqibnRCy2tAefNhLevHO_G)-FG=!?9i(Qwh6fXSJ^?+r<J3?s_z^_Lh^p`~Zq%I_&+wqwfy6q6'
            '@D{#Pwun-`gM(xVO#MHfwa2$!l2UjVBbyqd#ax|45*-^xH*Ldr$Kqz`B@+|XICo18rtO;Lm<&|IPQ`>hGS`I>X2-3{VRnVWHXoXM'
            'EjXQ#T<^|~c;<YA3>arrRD&z{a@u<T11T0dXMHN3ihunUiR;O-O&?Y)he8jihDXj>ka%jr6O#-A{$jX50#aP66ZDv6cxrhB(>mFQ'
            ')8}!W3s)uhv)D)dK$ww)J4&9R*;Zc5>#wT%jx2IJ9QKdj4bQ)rA8y3VNdGkxUI3Y0PJ)b7WLY~1Gvd(^K%Qq*5W?5pZG`YPPYAw8'
            '@gO83VVDZ;4(1gIGfHj<;+rl4^Mhg}#!yP)-=?AH<XuacP}vp+%;_(n;SCP^=UfT<`i07Qr}8~k`k`R$G)nxj9KkZPCD$R3Ek$E|'
            '8wX>`PqIqIEvz<GfuiqGC%LY~fUYc!q$6RF3S=NwQvzZPgTch()Q3OFSuQBJR7g7-g$Q%B3?|ga8aV)J&MHlp_@(67RBcBV_%uq|'
            'nAzhB`oR=)Lk7{@UjdPmL*HQ7Td*%a;c)+w;2o_$3$f}EJd;(5jnSP4r{S1oM%4_O>YOzcEbT;grK#NbW0WgC3c0;BKJ71j7)LHl'
            'Mq=4NM2;gk6^<cRXB&ev^=~R3WW6_E=T(L(1OU~lNSTd+F4ga=EejrK<I^B8*?_Fdj)DM6{(2}3;c=X@8YtUWUKnmvYelMHt*XNC'
            'RPKo$7vdOlUs_%Io!aIVouk-Q>vPp2svJnyS9sPsPNTH{CKW(+b=~>jCcJ&vzN!d%kzk2}dB{pz_(JH?xlGuwAlN+MO44)hW2Zcb'
            'Q{}yCOLuFx);K#%tR%`7s||N$%l=<3250?4w@0<txQkT+PnOF1jCxw;TcfNNjmqV;=rX_>zP99+Zf8|AC;L=ZGa<Y7Ieq<7gMs;j'
            'L^eEDSEzvMJY`yC4x*|f@fj7OiRAKU@`79yC5be?7HdftDruF1nd?hP*HxabEGzxl^L)Gj^#(KFxHk3YgR#1b^(K?5!8NqM6?c72'
            ')3QfLQ)0bJm}=sgRWbW_ty@hA#DPL}HljtF06+!<tx4HMwnA-6DEctw^QsmUo$^%ynQ+Ul!k@Ep{_)^!JsST{r@ve+&Qz7EMeb5<'
            'K`u?Vzjw7&@l<I>csE-X!yV^pYpasCT0C{jvTrKhx@CEU7{`v_U_CM0miV#$U2KK#h(>NzaAx_-e5cv8tl-X|;LMA_xy|FzDD}&k'
            'b)(qX=H89lU_8=w2CL9pH1VA8NYbQJ+lU#EQVK$KUKCii8guO3B$_2-;mxfAFMg1!KHGI=kw(nn(9|lasw`hh%lbVfw{@|ir)L+~'
            'uFN5(E0&V0ajxZ#yeN`(r@5D6=(~=g=NhO|X*d9kN5h~5)Zmaw(jo@&W<^;Lea?8=ZtH9SoUyf#FkT{bgCa^f1EyZA!K*cQC7XM*'
            't}3MT9Ww$Yc`6`NfMh&Qf{_&lPA{9qnx&5igs#b`$_S}ypV=khbNhE=NNW8jz0}KXJd4{`55E~VS<?m?B%mF^pztpLFf=GGs?fH_'
            'H%Pi16QU7c5*EJa-3vk?v0PyAD2%+#gle)aGh;J~@jAEFDBCjetS+-qQ_``Bihax%go$&>GDKM<K`fRP9}d_ccc!XwQSm3U;!<u8'
            'm4+pkjBomLR5|uTmg{x+SIK~d4;gE2VcY0(BNx-O!I2WmmWI#mkqSZWRPd?>k?bk|(Y>ey5?A_gsANHsRSZNH{lF&+tsDVc+N(1V'
            'mD*O6=%(c<rbLY_^lZrE$L-2H$##6bl~(bOGGq##E_ygL$c!`BE>KuPkfQDU%m@&m9;HA5dA}_)*RZLEIFH5=^{!jk4P(;M=sflP'
            '336lF6zZmZt`#Q9!OCgWi(_=m{@qWqpFA$V6T^5=S0X<qZK^=9DH^|S^#X`~Hk{59&>(3R;>YsgkeJ(5Jj}ONC6C9F<TV>ydIn7>'
            '@6Jx&3|<a~{X_ig<<aTE+au5&xE#X4VAw<5uoRx4VQa0W`^lKnL!&0Oxht!|8#Q;Q!HsEFE#Kd1l;^}48<am!SHLUhE;0x#zy=i7'
            '$d+!^<K18|E~*F8uFc0=oL#M(CyO-gpD%HlT%r7$*Lcl0Nw`<nHI;|2b-nZuI3vCRj?Ja~@XD0Z1>$2Yzquo_c;PPOzcct(V;NtT'
            'bi_NE9xa|ugE6oopyYJp{8Zd5p&_O>F-J^q<7GG@lO);(gDh2dLt+n;z$xDaToYwx7U;a=3vZ%Jfe6g4jxt6(53;=SO_0juh9sHY'
            '!}wW@M$b5r1H$L@lsrnfCke>Zn}bAwxjkU+=ur@j!{r2H^DMFD@tAqj7Qy?$q8r{IJ>;br3fPh3J@>h+DCmriEK1<F-sB|*O4!KR'
            'X>>p?tx^VuHg0gChC%N;!AqCL66GHD-VIctqtplXg9MAegGr107u44LqG7Q94v;WJCl|{9Psz!MNTShPg>n%of#E#EiA2GmydDlP'
            'E<NUKp&)MJp+N*8osk}pYZ2QJZ3iRLMxj9&^m+h(K;Qh-u!jYWg8tQ8TA&@Kic6co<}VlU<>Fv?an?UX+M4f-vYSxZ_VSeU=<{F>'
            'vejxK-Jm?IdBC-ncR0CNZBL$8gJaOGXKD#K{V}<NocQZAbUcG;OMAkjQ?Id-e^R^fpbYlsROclBIuHZc-&qLww~CXFA?N4~0_w=;'
            ';qsG8v&Pj@D9O#km0A%I&LT8Boc*IoFD*%k-zx?}?v^fP7e$5+#H}ecY!_33erTCa<}LQlbf-}Zbg9(u5+xgZi|$YOBZ+PbRi4aZ'
            'PWPx#6)PX<vQN<MMV_G^Z+74biTo_y@Z8wN>_0I5Dolnl`19@5HB92#ywx>kU2TY=LNjPV1))&ZX34vNXhhjKtIWo*cU}5lAb|-W'
            'y-a^!91|wQKoZ}J<`eP#_rij!<KXk%&X3{;9IV45VYYVhL?2xq@6W=Bl2#kgq<4iHHt<B<r<2a>*WR|0GdgY~2n6F0JxV1@=Re%3'
            'M=5jF!-76u&)~JnrzC{0ccuIF?sUJ+rS9WiPC8dxeQr8)hPksZVkWR{HPn;*97<@YlecEXvWu`B?Q?@e>H<XS#xvtY)(k1#m0ab8'
            '_PNeo$udgyJvYiylp7euF1ErD<v_|aU52uRr!?qDG4WcSi6$R`mSH4CHb9d=3xVP**EnS~X0bR12WDwhQO3buh-+iZn`UQ&Pvf{$'
            'LT_>n+rDCU!ZWHYczn$z$qHq-Pf@6alcfHZWp9xSg2>5I$4jjBj75Bc2pz8Q9-Yg?2PvnE#Ea}yW~v+k_F)Ndz-qF(t0c%W_oR=-'
            'AZLcX3@sgsOtnQ%o+}|O`XCVu4u(g+iu3+oE;w5m{DC6FSsw(2lXE4IQ7Ho<BRz6qsV72@tJtF)3-d)8-@`SqA&v<>3t#?S5HA@Q'
            'KqXiP5z74>DbX^#!Z_VLxhiqyMz-Wly@-k!Dan~dxXcUVn~q){R-VM%e7JWcc^AqSD$c~53Rx8MuNGIH<Rtfbfk__nP!9=WDAqaP'
            'Q~p9OnpDzBMI?mE(^q-6bau_!56h~OJM8snmjJ~Hk<B7rh+s~(G1Q{S4O_~~WD2ZFqoIm}wj{U{S#(|&=fv%YKa_65G(4Z!v)#R&'
            'XS+|ecS_EMsMw1%&S@8~dmw5)l7SaA-_#uFnRaujgJnx!bEq;k1S@@+$RbKU=8qpf=rPr9)2d*gLe^Wx%@bh+atKY^jcKYQo9a`q'
            'ifO?&s6rT?^OL@IlfUyNAm!OP?;ZDTPgJ?>W!TCJ_sZS`A2=VLzB}*8M9XAJ_2uc^(dAqLGNTnn@yq`CctkT;Y1M%?G8<k7XK|a4'
            '-*!u`h?tf7*wh;(RP?)0G7P@sCz%_?ZgrwbY2Q(-5OpT?6H!8^V&2My2Yu&OeND%9U&H@deM0BLG169V<u&k>M_tK_Fn`sp^nbIn'
            '{b>EI%MKv9prn5p(vteoQi2ixW3$+5$EV7ttqQTITYgj+|Fis;jW9zgxFU11LJ?{%?-UB+9T!8eQMlSG=Ul6}uT#$Zr5hNdmAZ>5'
            '4TOb!qxHkocg}DBsyeH;72Va!^#&9?e;Pa#<$0tAWjLrEcF`{uUbDfi;(bm2w+0{D3IH1tP5JF3>F10fS3cJxACtL1-i~X`mYSh}'
            '5RF$TieehQ&Ti%XDifl0cc{g!v0h6HvpGBM)o)GdVavJ!d>muU+jDwjTEg*|T&h)&O$3zbB^e?)h8N&l)7*@cdwgky`{Ke)q_kzw'
            'aSQN9@`ZT(_O~=IbEB^Kx<8sU3pEdz;<m)oiY#;~RcbAz=7AS+c!$33B*5cUh<qfUXvyAIa!vi#Dw4*_&2J))IbA-6GeC>qy*~!#'
            'U>A;lG_v^~56;gAC$Bfz*G(#%e$$Ph>%ce-$Ya8C{{jhoPR92pVYTx}YS!G%O1D$)F<--BCde~}6>nh}CS`JarQK)B&XQGZNmO~+'
            '#IrKpttKH23RMVKuDnbm-&||}@6^=k<}3_T7tUHmg4Q}0rE+wt9m`Z5pIS)NWQVCb<TzUq7cWPy38KxFU-2C&Y5>TQzSP{Zzb#Ko'
            'HMh8`OS=0=FTHl1aHrtN#N?%5fw8g%<YIXBYrR9(V`T+o3q&F0-+C$IqK&xT&#r!B%%K!w&ZcY3yrB#q1p->KfK-Htv-;4jz4<N{'
            '4cD7W?`x7jl0E&LMmpn$Y*&}P$|JJ*#H@zE<`&b+f)PmhTbA^TjvU3=7So!QD8Kqo?i3Xv38t$O9WGCK?xz&u_SwYSKErkd=T^Ph'
            'v}K1Tsyo*_I?;|x{Bjvz8S=WeL9aZtan20PuyX<M-FjV)&@Z6aUzc)MQ*eT0PPFAt7io6>NA<W5&e~P;Oer6Xoa@@=rf7l&GXLIH'
            'evXZ`9ScGV{@CC{rf|}%2ixn5U-~CUgSP^Np2NYuw(e#AO#mZKfT(0aN4#F%`B50mC>{wugg3TcI`?KA65pu(z?8wXld1|C;*i9Q'
            '!P(&GxPLf63&ey%X5=P4RFAcOE2{OA^(<!HcwZ`(RpyP!MYO7emS=_OT@2r}YGqL8e~;k$GbPo^|A$A3RkImJ&CJ9kzDbspG~(97'
            '^aq#^g^a1rekmu4>{)+`8v6L87?6YMRxz?dfZpfLu!2oz?GZG+H*fuL!Mt``GA_^<x19&kTcQ7L@H<bU#zS3y<|?mI*7%=#h&8u-'
            'IXF6k!|ML!;&j;ipK)UAdw<qRmI(=q@;de`JzO(FNgqH%W%1d6H9P-jAw79%Dm0Yt$ZI8W=dU~t9?Bz%=17v(5F9B1_*jW5B>xyd'
            'CO8Y8>W-zbsLn#3-et{jljTQ0%z_0b#!kEp)UK5uVbxclW|2Ra0tJW?Mxf!E;WB@gcnMB{<&_(sC}VE?T1yW9m(qO=apk;!_Dlav'
            '2`t0GasTvU$d#zdO8cK`+$k!jQtn=<Ipcp-&QoP_z}l%>$kj<T5KuGRybO2ldxh>E@c-4YESr038HSw-@PDEAdC(i2{L(ub9J+vc'
            '?=FS}y$X}ipTypN&6i4TYhk7Drp0#~-2_wNb2`4|byzm0R<+4V3(S?Af-bhV3h5iA%Z8nKUz#{-a|)H>FTDFgVZ4pZ9F-;xSws5E'
            'E-0-P6cR3NX5w)E%gxI*1;ol1RJoWX|J+!X`f?fSOJt{qY8XBuE2(Dk((;OG3!#0mvXym(N$ktJuWVS6+}2JgWyw#)J6UG0WMZwu'
            '1LTu?m804Bx3nO6)q=jEr2r}8U77M|_o%M8^rR<Q5Us<{%l-D=mAaYqE)mzRuD0r=f#O{<i6t*o@MBFbiBI0;Htps1)dL7d9^Eu-'
            '=ZFg$Vk4icT0+^PNSCv5H1=l!;|4n3>pM)8%^F*eiPNyWt<6N@Mq}$+$X8!OuBwrzKAQ^tevQv2KV@cP5F?v0ZEd9OeIZ5qTx#@C'
            'zC8Us+gziTm7!z0$<%9on`mv*Mz)+=7x=PvaOx!i=JRQOGU(HG2}8E8oCm2cSs~z!aWKmQaq;Wn#o5Ktaqn=@X&8A3e-APjkoc?T'
            'L(9Eu+tSZYUJy|TF}c@%V1HEUMt9yn?wt$=2j|};Kbp_c!dzOFQzDYfFopFR35~F%J&?52aF_kJ^V1WVMRCVV5^qHgtnrj0N>zNZ'
            '^u$f$In^pk#eL$jZ(NWuS&dUls^N+2>ua2am}1dxrz6hrMlKbV?#Qaht*mmzo_peEiSD~bJO;V7$kpg9x$^IFo?LkaSoOVTV-%-c'
            '6k7(($OzHmXzZ6jWydiXsKc={&7=XVYyP~C3v_<|E&aaP`Qc{=o&jOwe0|XNT)X4Y0Hqv%^t6M)Jd5U@V|`K8{=*oMW<XpkAjOsD'
            'HibjoY<$AlTjTG`hC>&W0fwx#_nbV@B>bV>40&K|upyubNSVjzF$&|GCHm}0sQE1DdmngW3D4v~&rCkJ!m0fr1rrQ5f^L867K;_G'
            'ordxGnYZwwxi=3|wRg4y4;^@;k~YrdlD`EXzOuY?>ThM&&1TBe@m2|0z@=CKLI;chi{~hPBf?hS87uGlw8_$BD^C={Nin-k0Nwc3'
            'AJ6coV`N6<js|rRy;nnwLZ!g89Z6~O*V3#0+;Z?|-(Z?NdOTpJeAC~bUhj+RPYp~2L0*83#=|v>zNySVDx>tEKde0rF;8ZQ=Tuzt'
            'yss%GHw^suC7qjiN*$Y#hd$gOcTLDoS3<yH-{&}V;446fN~!J&IWra93i(;r@0Q9W$19Kg6#g%02`o{M_jVX0sHW3#U7V=vK1k%U'
            'GM5&1%ZmQvMl$A)N2?0)tDB?D#Vyc0Dg0R)VDbwZ^Iwt&BeL%fLj<)ta!D<&%z1DFBfpVZbDdstW;^LQpK%98ew;6!Gf>p>j1hW;'
            'jYse>rWh2l=ywv&V2Owp%@V@Ymt3#(tXRJSFM_j+-X}ARK8ZuI{RyZ0uuXSf0(esOxh|y7Dis^~ypL)D3O+@jx3i*1%~J+j5^I;|'
            'ocviHCm<#bs*4~hQP2vaRV7`N-Cy(*XEV!MVFhaX76{a2zJUbSOR@l+ILSVS%aeI*#oq2W%3*u!J^ArDgG(nT)+t{eOC}6cyA6X;'
            'hev6SFjK{?|6vl`fHKi^)}|V6QrPBbmHDajMLCBt1}Dj|j#f&hv8w!;YqA=XOaM#)aIV={X4CdhHli2SM!>NaX8~{u%1aZ2u518u'
            'WB6KOH2xAr@R9o<_Qcd>BRJSVIBtO)hQ6JfhVr~1y$V1rYjO_nec7@D8q%cs{Kp@kK8F*B+hCIbo`#B`rjhJGTYg_&oy^7|Z?+|J'
            'HdTNDsxd0V)%Pu7MdL2W!#d--8|#i9n=568i>jhyGQvhRc{C%3Y|fCb9FWz@%?q3fDF)}dXyvdv+mf3uN0{IOn68}?WGxsdbA@}m'
            '1On%J3)ELJRg@Y3D+A-rboKJ%>jFx>5=}vL;+RNQpW7<YNFH6uQIMc|p%q^TnJOiEhcJKx>~;{fchH2`mg&T#GcK}pPENuRmXu8%'
            '%cCr_+c*h6vfy|Nn2VNf&+rX|d0d<wwODmSk4|56RVMT-#8I4No*9k(Pf>J;Zc<Jf;)QrHR^lGL=Vb&f2clN`C|)~q;Z)3Jx{Nr|'
            'i3_m^A8Vg(^Uh#?E%+E(6(*d^!lf^5zo1+hF)xiQP%Ns_TvZJ3HF-q4i(uhny10(oZTZ7EY0h;a|C(}Wm0@UI6_19ZRml%dr_PgS'
            ')eMqQj?S~j<7|;!OV<p9MoMYEag4+o)=W7yvjkp)lQ&tQTbriYwslofC1Y!L%KIXKd1U_F|M&v70u$JPx>U;h2Z-Fxyb*{*fdCz9'
            'KJi4ec5fUM$S{bc6dIyuWQBnlhQ~4}^-Bh3eLz^bE59)VtU$3`#6gsiVNuax<9+_Hu0?goT21s>c}iazfL1q>YPM-CTHQFG8I)Et'
            'R=H`#coi{h6$hx;$XOHPMhZxI47-`*tc`+O(_O_(RmGSU&holYGxKlCCsIQ->ch`~4R{Qb#(rLh9E&Jf6~U_uDivbp%3n9%w=ky8'
            'Sr^||?qAH4w@LJ0d6b%_&IU1pj}OKz&9N*IOA1D~S%&OyHDS}8`J^0ETp3VV{)Q!tQ7KIOL2+1{(wj51zu*{St(~J&i0&*1X!>B4'
            'Us*>f@j<5y&^-yZ-G$EtW8B(TXZ^lA9Q4oJ!``rmmHq|?eWPD>oHv@Xw42kKJy(^%$k}Y--}nh#ad3UGVeifH;N);{)>4JRR(>`E'
            'N}kFj^AluEDt1$;M%7Yg9<C!<*JZ<Q!O662cBz@Ux+cpXaiKJM7q1_B<$zln2$z{wv@ao(-S?p;U6fiweZnZ!;u|RN@BE~lvs4rv'
            ';|L9!AbigOpp#T9^et56=2EeTo_#|gH@LE(3T~M7<@EgtnXe|Y-W~Dy4#=wh3bQ)kVM%<H)nPDlmmyM-*r_U%uMLH$h29vmOyDc+'
            'oy18qP<B_W0fwU#EAvI@b8XUZVu_$r?xM&c^S*J?Inpd^^EGai`f;sQsy63ip4qHht-O8zJp<UC-FwMRD#ce>W4T*X0E!od7ZBC&'
            '%PVzivk#WdGUX=S@-v-U&&g|RqRG83O^}s{4G$!a%Ih#qkDF&P>{etkEDS$wd@7Ik@LTj-^ohUxP)wPJA$!%iLu*n}@Cu6vRQTK8'
            '^a?aAKdM5g(i3~!9Mkl#R%N(3r@NzV>eS>2<y+BizCs1nuKLlE5bN?k5(|`36Ytm`G3(k!(Soy1b6rM76bS9H1X;GKM`M9NR;#mu'
            'ANQ*Q9i=heI+Rv@$!Zrlyd^bas#R&px^#1#Y0gf4rhd0dG(i=oTt;f7+c=xYbN0zVZCz3XmCg2h*>RHuVbdT2#sUMjc|^e@iYC`Z'
            '>$M6wHM=O_Ea%F|apsTx;LcBljL!x_kig&4W0vmRFvZ~{zE8!DNXLL^K(;(`h)>5Lucqpz9C?lkzD3n27R}U$2^<}rzIV@h?_CT{'
            '>$$^ISIbL<1g#ajRX`NXQ`z8k!f(0P72Oz_u=Q4+(OCAwsv@Xot>CpptHr)`p{G=8{tPe&)B*W+3Y|$7m-^k6gYIZjMszIPF*sSH'
            'xZMqduYF&OBn*8;$uKj}@5K_ipZSOyOY*Gj8*YNUj{V#{kZYO;5ny9<ligaIfY8N^*m4KdH!%0L_C|29F!-~_VP1mXW<Vb;!<j1s'
            'o@H{@-vRd9;O4e{7l+HaZ|1+k9CHfvFUtosh=4+cN|Z>@HzA4dg*_|i7G$Bn!zB{PE=oq&%)2o*xMiGUPT&{%UbI{Q6_whn_F|Up'
            'jcL-|H1LNz^aqZk&qR^>t*L<VW}9;tBpy(H;9$uj@w4B%JMiG(Y%rvU_MqrwL=NZpXG+cpf<A8McTyYZP~JG@E=2NlTI!7{Q_0Mm'
            '8*xi;saoD<fMJP&s>e%zAR8(yj^t_SRZ`0zKNDhAS%3MyfS*W^94PK`kxEdQ3Fl;T>by7A-4O#6OeQk(ohQzZfpdsxIUfO7aXaFL'
            '%s{CYp=*_3`Ezf@0!=bUU4{~md)Ox!Q(DDkl5n}jAd*?|?xpxOT4v-a#?B|6QIC-(@@u>tE=N8dmn0aIq#}VZ#Dkeq>NHII8>l>Q'
            '=UkZg;_N8h83(+JDJf6#vfAfckr{dl$mwWp^)Y^^MMJ=g{JK*o95b=0gcFshWO_5|!CkH&%s!dwioNu<Y1vjyhjXWH+`lggP&MPq'
            'w;PXZ<21X@TaT>P3Jmc6&iAr{BWdHcLPsyb0ORxCVgH~nwntcMJz4@<;H}lIHzM=P1P~(F#;fNgnq73(>%{UQwQy%fRvQW%v=Qhd'
            'd@dpk!hpR<tl$?G+w314_y1P#I<-AVP4~)*-7)<h@NkE^Y_|5bObuKmc9JR=%j+D9msg#9bj3DQ6@pjnW8|Ud<ut}qP6&OYuN1vN'
            'T)$S`k5zB6!#8mH%>iCVzxPMi0)3xS(Y(&T$;C}z8u$LFvrK{=$@${B7ev=mh*Z@^F0b-z+}cZYOm~=+mNQRPu9i}A@G>D+>-8Y('
            '%}8hi-}L=RRt&`AgU<R|A(qF>+Ik_HPwNsWW}%Ss#xzRBJJp;0^%dSM_X7DTU4z~H5^qO!$s5=qW1`p)y28Uk!5i4bycxX4+A~Q|'
            'u8M=x$F}e#{0CFl1DN+8ujR&BD*dSAO(sp~YL6^8ZB2+0&SoFKSrzhw*avT|>GQ)1_o4=)X|cV0>QsgXNfeXQtUc+PUamdqSUKz('
            'J}R^9t6tYe1s+cFjNCCzm*~q_#hWC$qZhOyl_0|fbE*|QqTH`znGzT?s#50Vdk<&~eb&#5lgkP4)!HQpC|4VlZ`%C8;1KwU)%qGv'
            'RVRmWwE(DUL<i*Lk4X1QwW!km%NGjQsFStJvVk%~i{bEG$x(;SK1xFUcRdVgm0Gtp!AgN0CxH7z9!=z!mk4Q5eceAEoD5tT(f#Fg'
            'aL_;3EU)o5e{s7?sJzc|Ji~tvZoP!$&Bj~2hJt@z`B4&3n<IbX;lHQA@9^JV?CUbSjRQcPAW3dx>eO3ial}9SpiPI~gu4v9g`e@y'
            'i}?sv98&A>Ch$j|8tM?X7&KMt3|tU&X4%M(f)N!Q@=_1}0*#5L<w1*yA5e!LAXyT5i5jaHQgjr)I!FBTXeF<`akBit>khF;0=ky$'
            '9wg+G@HgPuk&oY8jsi8>FIe7S9(cgXsrNac%=mW1pT<8zP`rHl#Q8GxrXXOnwwx`McwcV;2DuP?tyf3Alh+r$*Yq|^+QYNP)_42j'
            'Fdi@2)~;k`#0@qf?u#eS#O%$-me_mV9tC8u#SGNmy?A$UESm6p%;o<41UMvq<h*zM4oe#&Ds`c|ySuw<zIZ=49KOLf^zh(K@8qO^'
            'gebFDm<-N>?1&$fZHPZ`aC$QA{}onr`nG?<0oc&{=@a^pwC^h_M!4*f<DXqfEL#G4U($<~;a??>-#=4%w;)1=u2kQRAG-x!Y&t<w'
            'gAJLO&HA}3vXAoUSY4)~WnOfqG?}tcQ1+c<?r-}}R?H98+vm?Zqvy{iJ~>n3^Q!D_MDlpssY#3Gz4d6c1>zh9egM9C3iM16??`l('
            'F(8a)i}BpeV)x#=lPCrc_Ju+!4=JdN_VRAwT%b`Ky-dQiYo37`mpwVqeK5&xyTyaVVFdU^&57S9-XhgU6j0C;XM<O-cHZ}X>F1|a'
            '3f9JnmoZ4vOAbRDaDWyw<LeHr@)~o9$>N<O*Z`Eq7ooSp%VqXPl7$2)M#5i!ww+((u?H&KeUNZuATEH4bnV~=Ks82d6)M}(gOAn('
            'YV&kT)o-xEKujgVXb!e8jcp0gln_vZU3{<M3_ndU=0*QpFAQ!_qF&dMA%ghVKu9*k4{{xy1&ATd#-9d2ke@z>$7Hx2&nfC>)If+&'
            '=b69`c}3<z*Eeql@c#{@ur0i)RTHr=sFkm=8cU4Dw5$cddVX@O?~nLcS1v-&-||l8<^O>Au1h}#(<zJSZW?^#gQ!uTpX1dDyIdoV'
            '22n06o<k1~u;V*s1gel?{1oLSEK2mgcl35}@|uemkivk1oJ8CAZ+<01N>98R{I!25PG7w`?+@u!&Al(z!N~8(pFL~i<kaGT1f=4c'
            'l|~O$(<zo;c>0`5&0&E<(&g}K9GA;X0)rwLO`q+ypFLv??ueJmj6FgFX*<H(sL;QGkHT4rk*TsPctZ9XIhIeQ6M!K4z&Eb3^5ON)'
            '_2A^y>9s08%8om7l02{JbiVL!#3TmE4^&f{q^^rM4L)EI1YQY+V9tFH9oyMf_WtY?pv#cuH7;UcLa-?lS-^sEAGQOiWJ<;{ihRuC'
            'Sfq3G0st{Gg~NyzuND#nZ<+e?(_Cp>m!!(C{kRq|03xN3s*ZT?g)>>5k9Js9=p)d{P-L|M_OX{s1IgjZR-W8dKK()bgrRnb(8K|Z'
            'n`O4rK|g$P$?f-bw`X}!5pFeC&%|T0jEsX~bj3b-_n)qu%UvrjNq?`KAB;l!j{}e(7h0XcFfyeo#pp1PuI?UaKFerKj?)C*`7+Jq'
            'rWHm*=rLXfRA?j@-f2OI#1iXCvoZ?B7>6DsFtIt=bep@a9Fn3fn)z}aBQ6OR!l;U7NH`6XHW4IL^^l1~BA!fm4rA%<O|Bu{gr=KM'
            'Lyg{rJ^X<?{2*vH;$Ov{MVoeuk%lbs=@1FFN(T`$=<eR`lV|AKzF=R-H9We)P_0<!4j$g|4$fysos}J+D{Z{*g9gAj=~habnAtxV'
            'whh>UBwz0+u8_8bXrJ5;*cm^Zwj@K0d^~o*^`=u!@Qf#ixun#Y=`vMD@lxHNAS-Z2D$Ub{?`5|fF(RuO<DD6mEc8FXkwb?leA@|9'
            'h8&8d`u;L^xzfS|iad2W>KX(c7iFV#@BeS_TASN8vg~*L3IzEeG8QQ*Gx27mVWyZ79jhWsF3HZ8SIZ(n0-`v4E<nhpHT>^;U)|_N'
            'H>ii5nc1mLr4o^BH2U55aqhV<$%LVvH-@`sP#(Y!h>owOy$jVS_n4mJPFK1SR0%X|${oUf5I8jD4y)Q}{)z3|sE(1FULcRzScB_D'
            'F6ZJKP>QnN5{Z?ly!@~+!;yHnuhm$09U~vMI?F6{o47TP<Y8<ycM^fB{FlcFRHU-U<rOcoHFCu*rg=v69Wq)hOVfnoxhk@2-5knm'
            'bI=t1N+E2k9_ck=6$yTV+tn(k=C@R=RqV_n-=^^^+^M+c>8w5aV;4_QC4uShe>^(;UVcGw%YgZ)m)onhD*-MAgpUslg7;QRUp@L4'
            '=h|~0XbFxQ^!>rnA6|BgLy<t7a+7$)radK*8PD)6C{*EnX)#(&GuNbUeKr4{yI%1dgQBAtHhUYBBf}gl9P~q)2p$*s^wO=SwCAZ7'
            'Q$}yKyvMO+UN*}UQJ9j|)yqo}Xet3$%NYFqfb&#V>IXII#r{5CX?AsLOIXK7!NZ}xc6eme5V)roKCD}*Scv!xKqeE!$I-J*3Q^8V'
            'PHyA4;6I*oUKnQu#qP-42?r&ohp^ygV>3>-Dum}nfJP}MD&m_Dg!#K7=DCRoROpF16z}RA71g+$Vnzf8Qow_~)hesWM@`=$uH{>}'
            'IF(oVj|T?_t{wUcYs-R*^Ro}^p;4HDCW!xnOd;v~hJ^VQ3ZhYfnMtOJ(m@!+D*2gbGYZRsVUYxX2C9`zR4`rI=q&M`X7g26b(7{o'
            '4mu8k{d3Fco8$k%GF=@UOVcApY*q{Q40#s4LSd0UK1015nFuI=ze(d<OT*e<{@l85e)E@Z{lBvp!rM`|BKomyQfpn5T2&X(S-_Ij'
            'E#?i$g-fm<YM-)5OA(E)v5V?1sQ2ucz`Uc1!bp$IA2z|gdU6w%hr0A?^M-h^g$f5rC3jp%1gup)&)E%SE?Cg?dRhd%Ta1S~m#bs0'
            '-IljCnvZgf0~s30V0}j2Ep^P1)`m`?c${($Bb`<@bb*AAfoNpQB$_abqH^Zp2dXIMJ)y!Ik?<O=IJ6W=Y3|zt*jFkaCm|%PIYVQN'
            '$VirJohPS@#WJH@ZxUt7%{e{4Iz8#o9_{<!m3^INRP0+AQizG?g#vQ*5d?7%ow>F&X;6Pn)}RDYR17;-X9dIAB{3ccrm}`3qw=@K'
            'KqWG#?!h~VdNP`-+%2HRze1gXRqq7zQlh^Xq$Kpy=xNmEg#qRy#p`pPA&Vt8CZB_JlE~n{P8g-Hvbd6EP@i`4(jx`P*DRP6K8ko2'
            'J`N*cu_Tp)dq<Hb+&KvlDAG}<GOl)ZRHf0fQGvD`g8TimiWw@C>3BeBV`$AZxmjVXIwNQr4Nlm2OZEyC*~-LbLdmp8TJ9jZjgagv'
            '<(#K?oJ@2bkRVPr2OM)Yr9|%KFrG^|tsW|YU)@GA4-o2Q(_~gMOUS-gdYAfKw|;FdBRXwn6WEhA{*Wo!O>AFJSUCBy!6L@o*v2VS'
            'aj)RriiJH&hD2J1t5qgJu^x81BH5dpgv+CGl!Nk!UkZ9NjE?1Y9Hkb&v#*khEPa(xpzqpOzxFOK-oNWyT~lwmK96`g?2~G=lksiM'
            'dshS>k<crOr3xIP?+Vr-t@o*UpaWmIZ5<Ekmhj`75QA~%vt$iB-!RXEOoogsRPQ(G2ISSr=cZe8Jrt1rGEHvApK#51j(@dkP;A3U'
            'hmhnlU&+!k69F-l^K(fq6!Ec>A}3z<r$&27O-%%js>GoJt73pm>!DZ#+e=jym>{F_g7(N>8LEPj3im#>TJZl+A<X!S84s_+_WP65'
            'i?Hk47Rw|-3>Z}jW&LRZLhEwJ)op(EU+5t%0HbEuJqZS*ta{Eb`~}+B;Uj(6Y}{H)Uu}HhAch|mG-2ff-0Ed5<AaPskLL^`ha4U$'
            'sF8>VE?&$*cZzdEU)IY<L^<PKzQlXfz@29>1Sx}#yiQT^sn6F3-6wX<=oa)cbmMP|ns=TXapDjD!u4M-;X1v6C~{pqEfS@aeok|5'
            'e#0~nZuUpxjO~{5y4hunGuKnnVQ&34Xs;Ht@enQOm_o1dF~x}6Ia*`+uFvxHj$%`g79`4UDK~c%u<<)0HXW7_q&}AEL26`ylT)j+'
            '!S~}LpJGf)5)s;$>Jj-4r9th%&>my7!<C)LWeR4+(>Plyu|mrbME_2=^d~K5H2E{0>?mUtJdg5x69+&{do}%55l*tEK+9{oe4)tL'
            '(OYa_@!8#Nfu~}I(LlScU}>6*k5*Ya)VKitRbovw4_7ihn%`JuJr^H6QE?&aMW+`yVdxkaEg{6zNh7A5Hr$TOu`MI#KIIw9Z-c&P'
            '!28JT2lA@8_b6s4K<HdrSb>hU4^wF?-y&{3pjZ&?#VdRj6hc>U_plzMOjb>+8;)75W{@!wx>_}Y@GbY7x_oOsXOGNM*kuZ`a&5BJ'
            't3;y3drSo6;H(S5so?8<d)fKzwDX_GET9HNz6&QWqWBpeT_UxyVA0BMrTccR%y8Z9$!yChW~vE?&AQdshI6gWjz3mVs8Vw;l0T57'
            '|Ah4=4z?x|TMP`Mv&pD!1|nNDL3g0|scvAvvRl;Nak33>oH|%?gu_f~Gfw1d>nD#n%S)%TYf14~{l3(Q@@3|e2Mj0L2u22V8aC57'
            '>8<iP#uj5SYse;JVo*(8e4KkRo5}!_rxTc0)O$r%2x@(6SX#RT2Hfq}Rk$wfOwemZhf(T(VSXsoh~QGejF=I5nIK*03N0$^?Z;b|'
            'R;?IH=2if33<|CC&fvS_WT+f30ed(27A{QemYY?@|Fqw}y}bCXeRian#>y~8Y+k>#x%(QiKvoneyrVS-_F&6#fyfHH6Y=nbjRFP6'
            'H*1Ahkb$489>J4xFfngfUvoN<grG?-`e;<*AlR&cOrWe$SM~_ai9jQUQ?buDYD7Y_A!0m-Jw9gNzRpcRMbX%?TQo*&CN^c1GEqec'
            'KgnQ=<#WY{5%g_Njy?m7g5@;NRAkeRRt%`!{{SP19Kl<x>;Fuv2e<0~sM$jd7XmeEJ?ND29)5K(JYe)!?-}-oZ3Ruc!vnk*Ml<XG'
            'I`)|1V#*9$U%^_Us&dRFR&)(drjy$HIrfwXY%1o2zYw#r4|wdiuRPDj@&r4}b8RiZkG;iXbMbs-JIKDo{_+?b%vZ9**o*TC#uyYx'
            'rx5|#tXR5CvS6v;nJ+QgsCFr<%^(zawf?34m900PH|%^J-jIJ|gUvH+BZWPr+O+ZiRtwi^At9CumBk<HKEsAwsTHOT>%#pq3&%go'
            '0>FuPsx|-M3KpOdWQ{C$obqCvX34%vfr?D8{8b>CvxxCkCqf-1Lu9r14D)~~PmK!KNt{gZ1QDU$cpMY0|906vd(-}}e;EseFDRTT'
            'un+;EIfU_p&DHIy^xx?(O>XQ~;7|%zUdkQ<MdnR&wP{Uzs<ooV5Tq68A}Y9A6=DmB84Qyv{kc8;h!`PPeXPkE<>f`Ttv-)*f4}dj'
            '5&goC9&wGNz(+{yc=h^W)l1_Ps+B?(ZdoIY3Vl#(ptgdBi6)Ba+CtZCPz?Ji+6%UOT^$p&xAaw{7zSIRBWHR9zt;6GiG91>y0Hi>'
            'lF*i^i|lym*jwe^`A#E4gW}G&yv)`xTFekuIbtR+b-7O2VXYFiw6WaeT;ioPKE`taPGrp1oX8myBdY`ATH5FE-@kUwI&V5xr@_yi'
            'U(ZfITwIpZCWjZAdHQ+m(q^yyo%Yc9`ZlKLyS*Kkz*=Kncl>_$avO{vP4ZM0!N;-Z3BGl00$;vap|CTt_B~AV>oXhH1y&a}LX7+H'
            'RIc=m2VP)2>ATk5=Y6FtMrGg4?N;M0X4La7M*gCnZ^WN);s!JUtcJH_30q}NJ{#QgRxdNNPB_bumIQvyVQpu-!ER9=jY&PKI?1G6'
            'yTJ(Yx*Yrft+<7;OH5<7b$;urnpLBLukcN}e=x+xbbWOAva95%>Va04B<r(9{0yTLCwDzfhplr?m4YA2aSFAXe-ju&XUW6z2)~QT'
            'hP}vEy*!J1N(m_=Vu})s=5k^u!vD$f2w(Szv#)p9`@t>ZLC!qfEwc?Qw7O3z!VrsT-IOBBNq!v(($kVs37hGfuG*}pZrXyu*qX}C'
            'RzBdMGaDpvOnD^4xR0?%&!V<f2UO?H&z+N#)AL_?C#P>Z=U1l}=Wu!d$IBl*CEi?|bj|{1NrlbRNK|tg--kOGk?(#_4)XaT=;Iwv'
            'LErsjMoq%!V6;edfD&7VIQ`8|@lG;i!j*@yn^&6bIaK+Udf9k3o<=Dr4G1OGi%JAn{22T_V6jrMCk&@~wnAQ`QSn^yol~4)k2t%r'
            'unw57Bc?IYwErHc627O2Y@_;7`k9{FxcFf`tekr&)5&eboJNUEwS?>&p>2s?;O0kYD0;2fEwyEZY|lr7+Zk_i62eNEW}fBKDmotA'
            'jU!qu)Tjp0JDk4@RAvc|6<1<oLvgZ;bPkKU=w)U(xgtGj#2tr7erA3{pHjUU^xc)+7oWyz(a>4fGwb40Zz6FV_5jRBPWTRs-;hE$'
            'mmnA<@v(-^I(`>AmG+8CdEymtkK$xOU7-;Vu3onra2W{+RrHaiVVyCHCMag;E=4``OH)Q(lv08hipW3%G_Aa@C!JL$8S_XABp(EQ'
            'YbV>nTDJIIii!UH+%KcWEC;y*6lzS*Iwz8by-Ej5mCt0m!^P-Fx4!4~v$z#M!tY{XH7m|Yx{S)F0uh#~$I*9&u|wC|3Xi5LoSQml'
            'QAPfDPUtnup6qb&vO&{;JRN>XE%hTWm4>EwL?nr0;25asFY6XU@fikT<LGHsFh3cS?Eo2z-=1JDwgxNENqBum^g9rs7im_oRVl{~'
            'Bkf@lt6IR(VnwIu0q6~X@2833-=63V&<%jbPN1=Q$^LD`5pn%>G$h-z__SAgFDE0x>0q&j<M3WrzmY7b!M)LljT^jXw^@c>c`mLw'
            'Wblr+5CyYwP2GAqTuPp#eiDg>Ec7S16n5P1-CI>{g3&mQ#{ry<s$cTfjm!ftr?A42q*6-qL(la@Y)|2p<+GaKE4$)Rlx*43rAPTQ'
            'tQ+J>ZVd~jrj`a7I_<S+6)c~#RKEOW)e}^h|FnwmP@vu^0oB)5Mx3g1+O2{jZ?wstiWy0+Z86O;_|n(>etNRfv<r$JvP%_WW(~uX'
            'NwA|mab!vWk@l$JAK0u&?}Nd%OlzU@njl((0-t}z?7EXxL<i+!I*Z`2lqEh*(tvh-D(6kSptwO;cb|gqf;Eler@ep<%Fp8^H=K*2'
            'jMj^c(x13AktEe&`M{0*Ls4=dCD*!Ms&rtLq%q)XTfqU31V0NOof{Nxd@C0ww0$21T&mzZc~sF$%@=pcR4LO1by$x#@Se8t_VnRn'
            'PY=j>hd~2@8Zw`z?i7t86!ml^2>#3mQ$hgl7daZ`#u91!uz7&)rNxS@K2jTVk}^9DrMbco<~hg725}BjFbdDj;3R>qi4r$*Ra14N'
            '@tshpQVtn)`uZ|qm;36B?4gY8fODWCX^Fdt-7zGF8s2*Jr<_jqxLVtUV!nwJrFlTqBDF*O1Ba$TZ?A)N_)^Uw=7HCVgnN24vMtkz'
            'OznurT1-Llqv?eH95ny*y&LR&qUysKZ*lZR!*r2h3>GqikY!1c?y6`z^&&T6IcC5ZpiaVVsy8*?EwOmAO^*1H(jXeZ^5hD2d;2JF'
            ';d^CeP^hFmpQ;jzi_?0mIR(Wu(<VUF$|ELKT5nMZ(BnU~jIp&PR&JK&Y7MDL1QVgAg_cc%7Xj6Yb6w_BO=0KMq()>pUAH9p!0ae!'
            '%&yJ4x@w__ouW5bn|-6VPtlypOR-{Y{P=^rHhxKyBnIg(Hn*Ly`7>j=VH&A%NN58AzQ#KmNVpdlSsUaDMXHKAe)oZyEcc)oc8W8I'
            'l@-B;F!6f`#H3zX7B1a|pr&6QxekXY058TdCD5pxdZT04;S7b16s}0M5~%$y#h?&+klWmOwq9L-(Igx<uPjv?A{zz0pb;D*L=FVd'
            'eSRYij_@kr<>sl^UH<ByuJeta0*etv6UBjoJt=EH{75MxU*@RTJ!&XGEv5lP45DtLXxGwAd-GhI&kl_--EiN{CDE0wqfgZyygh;&'
            'vnDsEY!SvP%5Ubj`NI#MEAuUvQ}&yH!OWO5p#i57E_pm<`8>fU&66l?=qBh3Q*9Y(id;$dox$pbl&IG|_e_ni6;%WlhS3$dX4s6}'
            'h6`qtMt69(iyiq@{fL6Y(=lPfVOw6jdS#l(suQH4g)%B?)a3p>vpy&L12hZM3_d;)wet`5Znc2AMQdQY<)iave-*U7*+pwub^Zxv'
            'U6bqdM3O7o0Q<}L=e_p%yVGAT+rOQ@`_Q}WTy@%)e|@b+rO)X+C};zT3}L`kyg6gobyz#=M4ap^*CFU4t6AT%Tg(Qai~GqeM!BUU'
            '6a(zM#eHp|WCXh8s#tN3)F8@s3ie9WT^oX}C-43g1iUs1eYg*x{(gBT`@tZOM+sUhsU95b_PdMA55jgP_yrw_1pA1>R#5PWqxEhk'
            '2sjz_(^Y0*`vp<aH>a)-2d4wt@XD5I**Z2%+oP@wn!bHeZQ4|>Cr>WQ!s=<ZJM@++jGeWxYHcwF2E(t?ELy{@IIko5&pcUY$iALN'
            '2w<8;!9!WNL(s7W?1a$uz2J5{9E?*2wHJ`~W|>z0%K+`(VJ#!vze-SlSYDe@sQi^D^(Q4cC>0Ax$XV6O-xI6P!K0ImKkB$;yH7M6'
            'PXe3`rzwMcHB2%m)A;8_irRFXr<f@%12F<b{XxFYC%mJARX)MI$2hZDG^sl&#%15O{2kSaQMaa`{HZ0?{1Z2spmQGDUN@ogT1K}7'
            'C*V=S9R-71?tkl^`?Yy^SIPW9jZrpUFVf?G>kY-jVs4|a;d6L4>jNbu@;e=zgqeIC)DHL5(r;{E-5!U;R}0fuuXu!=)W9|PHxHRk'
            'cMNBNHlaQ`@1<j7wekiP`I4-~&<)h?x1q&al2F4hk_0_aD1|;)daDYSZo*!zTlEp(*V3VU(mC&Zxayn*ktqX4Ao}IUE?)ROqV#4c'
            'LcvTUwpiuG-YXx-gb;?~IG=G&Ck7VJlF$00&5|{o3hVJ4>GwW62XZwQ*PM6_&ouG4B?0*H&`ulHS-HTr;5@CXEE=H?KXj0r3f<|#'
            'HZe_tLTFZqOE%c@LBOJoKfHEupBCu6C{`AmORKdg=-mfqtDFMm7Cm-{{QjxNU!_UXz?}5V9?NK`V3bE`tdflXss7bdbyM<2L_54e'
            '{OJ27sLS!s2!>AL+GRdRI&zYx<kmniyle-A#iy%Yfjy<nfbbaozxbY2RWy?6^p+ZIGl|yoXu?TmL)th+hAz+`E|Z&_NSqf)S{PHp'
            'h2dhkPKmWG4yb<om7;rz-yO#fWHBU&#xLRVlr=ulnvhj;xuOtiQFG3a;W>EqifG>VgPRBqAB~T<;|vMc_Jg<UE6kO9Le^jSJRJ|x'
            'MTRl|nD?omtK)?=kEZL4&?2z^ju|adorrgj`QBI2Y>7A!e9t~1^LDg7R@fhPrWBn;Y_!0hGal2cbCu_zBq{sne8={VZp-=0y3kQ@'
            'Z_Ml_$cM01KmraX(P{?6Q9hbKy#MvAbB+n|C})GNtp}=Vy9e4Zrq+1@zkSFj7->HV@azUzF2m$h(K?Mrj{J=R`!E8z9r4?vBuysM'
            'ac~<=rl634RCzmI28(H&kMJ^BC2LTHla+Pki$z8+Ww>|e<N2sYOA^-Vk$DCfQo#EU?aR}nt(lPt71Rt~M>AgL`?#P5MXSfk2wLG`'
            '8_#iq5xqMm(9J`G^>YrY3lxU2;m2^`j9N*r**Y>Y`jenl@L5O1{4WWOPcQ'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
youtube_data = _local_pipeline_modules['youtube_data']
google_ai = _local_pipeline_modules['google_ai']



## 1. Girdi: TMDB Trend Listesi (opsiyonel)

`TMDB_API_KEY` yoksa bu bölüm **temiz atlanır** — `candidates_data` boş kalır, notebook çökmez. Aşağıdaki heuristic/LLM bölümleri de bu durumda otomatik olarak atlanır (boş veriyle sahte skor üretilmez).

In [ ]:
candidates_data = []
tmdb_status = 'skipped'
tmdb_status_reason = 'TMDB_API_KEY yok'

if TMDB_API_KEY:
    try:
        resp = requests.get('https://api.themoviedb.org/3/trending/tv/week', params={'api_key': TMDB_API_KEY}, timeout=15)
        resp.raise_for_status()
        items = resp.json()['results'][:5]
        candidates_data = [{
            'title': it['name'],
            'overview': it['overview'],
            'popularity': it['popularity'],
            'vote_average': it['vote_average'],
            'vote_count': it['vote_count'],
        } for it in items]
        tmdb_status = 'ok'
        tmdb_status_reason = ''
    except Exception as exc:
        tmdb_status = 'failed'
        tmdb_status_reason = str(exc)
        print(f'TMDB cagrisi basarisiz: {exc}')
else:
    print('TMDB_API_KEY yok; TMDB toplama ATLANDI (temiz atlama).')

log_result(
    'tmdb-collection', 'tmdb-trending-tv-week', None, 0.0, 0.0, f'{len(candidates_data)} aday',
    status=('review_required' if tmdb_status == 'ok' else 'insufficient_data' if tmdb_status == 'skipped' else 'failed'),
)
if candidates_data:
    display(pd.DataFrame(candidates_data))
else:
    print('Aday yok (status:', tmdb_status, ')')

## 2. Aday A — Saf İstatistiksel/Heuristik Skorlama (aday veri varsa)

TMDB `popularity`/`vote_average`/`vote_count` ve Google Trends arama ilgisini birleştiren basit ağırlıklı formül. Maliyet $0, çok hızlı, ama nüans/rekabet analizi yapamaz. `candidates_data` boşsa bu bölüm de atlanır.

In [ ]:
heuristic_scores = []

if not candidates_data:
    log_result('opportunity-scoring', 'heuristic-tmdb-trends', None, 0.0, 0.0,
                'aday veri yok (TMDB atlandi/basarisiz)', status='insufficient_data')
    print('Heuristic skorlama atlandi: candidates_data bos.')
else:
    from pytrends.request import TrendReq

    t0 = time.time()
    pytrends = TrendReq(hl='tr-TR', tz=180)
    for c in candidates_data:
        try:
            pytrends.build_payload([c['title']], timeframe='now 7-d')
            interest = pytrends.interest_over_time()
            trend_signal = float(interest[c['title']].mean()) if not interest.empty else 0.0
        except Exception:
            trend_signal = 0.0
        trend_score = min(100, c['popularity'] / 3 + trend_signal / 2)
        revenue_score = min(100, c['vote_average'] * 10)
        competition = 'high' if c['vote_count'] > 2000 else ('medium' if c['vote_count'] > 500 else 'low')
        heuristic_scores.append({**c, 'trend_score': round(trend_score, 1), 'revenue_score': round(revenue_score, 1), 'competition_level': competition})
    latency = time.time() - t0
    log_result('opportunity-scoring', 'heuristic-tmdb-trends', None, 0.0, latency,
                f'{len(heuristic_scores)} aday skorlandi', status='review_required')
    display(pd.DataFrame(heuristic_scores)[['title', 'trend_score', 'revenue_score', 'competition_level']])

## 3. Aday B — LLM Tabanlı Yorumlayıcı Skorlama (aday veri varsa)

Aynı TMDB verisini LLM'e verip trend/gelir/rekabet skorunu **gerekçeli** şekilde çıkarmasını istiyoruz. `candidates_data` boşsa (TMDB atlandıysa/başarısızsa) bu bölüm tamamen atlanır — boş veriden sahte LLM skoru üretilmez.

In [ ]:
llm_scores = {}

if not candidates_data:
    log_result('opportunity-scoring', 'llm-scoring', None, 0.0, 0.0,
                'aday veri yok (TMDB atlandi/basarisiz)', status='insufficient_data')
    print('LLM skorlama atlandi: candidates_data bos.')
else:
    SCORING_PROMPT = (
        "Sen bir YouTube icerik stratejistisin. Asagidaki dizi/film verisini degerlendirip her biri icin "
        "0-100 trend_score, 0-100 revenue_score ve competition_level (low/medium/high) ver. "
        "SADECE JSON dizisi don: [{{\"title\":..,\"trend_score\":..,\"revenue_score\":..,\"competition_level\":..,\"reason\":..}}]\n\n"
        "Veri:\n{data}"
    ).format(data=json.dumps(candidates_data, ensure_ascii=False))

    def parse_json_list(raw):
        try:
            start = raw.find('['); end = raw.rfind(']') + 1
            return json.loads(raw[start:end])
        except Exception:
            return []

    # gpt-4o-mini -> gpt-5.6-luna (fiyatlari platform.openai.com/docs/pricing adresinden kontrol et)
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        t0 = time.time()
        resp = client.chat.completions.create(model='gpt-5.6-luna', messages=[{'role': 'user', 'content': SCORING_PROMPT}], temperature=0)
        latency = time.time() - t0
        parsed = parse_json_list(resp.choices[0].message.content)
        llm_scores['openai-gpt-5.6-luna'] = parsed
        usage = resp.usage
        cost = (usage.prompt_tokens / 1_000_000) * 1.00 + (usage.completion_tokens / 1_000_000) * 6.00
        log_result('opportunity-scoring', 'openai-gpt-5.6-luna', None, cost, latency, f'{len(parsed)} aday skorlandi', status='review_required')
    else:
        print('OpenAI key yok, atlandi.')

    # claude-3-5-haiku -> claude-haiku-4-5 (fiyatlari platform.claude.com/docs/en/pricing adresinden kontrol et)
    if ANTHROPIC_API_KEY:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        t0 = time.time()
        resp = client.messages.create(model='claude-haiku-4-5', max_tokens=800, messages=[{'role': 'user', 'content': SCORING_PROMPT}])
        latency = time.time() - t0
        parsed = parse_json_list(resp.content[0].text)
        llm_scores['anthropic-claude-haiku-4-5'] = parsed
        cost = (resp.usage.input_tokens / 1_000_000) * 1.00 + (resp.usage.output_tokens / 1_000_000) * 5.00
        log_result('opportunity-scoring', 'anthropic-claude-haiku-4-5', None, cost, latency, f'{len(parsed)} aday skorlandi', status='review_required')
    else:
        print('Anthropic key yok, atlandi.')

    # Google: SABIT MODEL YOK -- kanonik router (local_pipeline/google_ai.py).
    # Trend satiri siniflama YUKSEK HACIMLI yapisal istir: Rota B
    # (gemini-3.5-flash-lite -> gemini-3.1-flash-lite -> gemma-4-26b-a4b-it),
    # thinking_level='minimal'.
    GOOGLE_CREDENTIAL = google_ai.resolve_secret('google_api', get_secret)
    print('Gemini kimlik bilgisi:', google_ai.describe_credential_state(GOOGLE_CREDENTIAL))
    if GOOGLE_CREDENTIAL.configured:
        _g_client = google_ai.build_google_client(GOOGLE_CREDENTIAL)
        _g_manifest = google_ai.build_google_capability_manifest(
            _g_client, credential=GOOGLE_CREDENTIAL)
        _g_ledger = google_ai.QuotaLedger()
        print(google_ai.format_route_line(
            google_ai.select_google_route('bulk_classification',
                                          manifest=_g_manifest, ledger=_g_ledger),
            GOOGLE_CREDENTIAL))
        _g_result = google_ai.run_google_bulk_classification(
            SCORING_PROMPT, client=_g_client, manifest=_g_manifest, ledger=_g_ledger)
        if _g_result['status'] == 'ok':
            parsed = parse_json_list(_g_result['output'])
            llm_scores['google-' + _g_result['model']] = parsed
            # Ucretsiz katman: dolar maliyeti 0; gercek butce RPM/TPM/RPD kotasidir.
            log_result('opportunity-scoring', 'google-' + _g_result['model'], None, 0.0,
                       _g_result['latency_sec'], f'{len(parsed)} aday skorlandi',
                       status='review_required')
        else:
            # Cikti UYDURULMAZ.
            log_result('opportunity-scoring', 'google-router', None, 0.0,
                       _g_result.get('latency_sec', 0.0),
                       f"Google rotasi basarisiz: {_g_result['status_reason']}", status='failed')
            print('Google skorlama basarisiz:', _g_result['status_reason'])
    else:
        print('Google key yok (GeminiKey / GOOGLE_API_KEY), atlandi.')
        print(google_ai.credential_help_text('google_api'))

    for name, parsed in llm_scores.items():
        print('=' * 20, name, '=' * 20)
        print(pd.DataFrame(parsed))
        print()

opportunity_quality = {
    # ornek: 'heuristic-tmdb-trends': 3, 'openai-gpt-5.6-luna': 4, 'anthropic-claude-haiku-4-5': 4
}
for r in results:
    if r['task'] == 'opportunity-scoring' and r['candidate'] in opportunity_quality:
        r['quality_score_1_5'] = opportunity_quality[r['candidate']]
print('Skorlarin isabetini/gerekcesini degerlendirip opportunity_quality sozlugunu doldur, tekrar calistir.')

## 4. YouTube Data API v3 — kota-verimli zenginleştirme (TMDB'den bağımsız)

`YOUTUBE_DATA_API_KEY` (Colab Secrets) yoksa bu bölüm **temiz atlanır**; TMDB/Trends/LLM bölümleri
etkilenmez. Kullanıcı bu anahtarı Colab Secrets'a `YOUTUBE_DATA_API_KEY` adıyla eklemiş ve
`videos.list` ile gerçek bir video + istatistiklerini döndürdüğünü doğrulamıştır — anahtar burada
asla istenmez, yazdırılmaz veya diske ham haliyle kaydedilmez; sadece `configured`/`missing` durumu
raporlanır.

Bu bölüm `local_pipeline/youtube_data.py` modülünü (bootstrap hücresi ile) kullanır — aynı modül
Notebook 05 ve `00_youtube_automation_master.ipynb` tarafından da kullanılır, üç ayrı kopya YOKTUR.

**Doldurman gereken tek hücre aşağıdaki config hücresidir** (`YOUTUBE_CHANNEL_INPUTS`,
`MAX_VIDEOS_PER_CHANNEL`, `USE_GOOGLE_DRIVE`).

In [ ]:
# --- YouTube Data API v3 konfigurasyonu (BURAYI DUZENLE) -------------------
# Her biri iki formattan biri olabilir:
#   "UCxxxxxxxxxxxxxxxxxxxxxx"  (kanal ID'si, UC ile baslar)
#   "@ChannelHandle"            (herkese acik handle)
# Ornek/uretim kanal ID'si UYDURULMADI -- bu listeyi kendi hedef kanallarinla doldur.
YOUTUBE_CHANNEL_INPUTS = [
    # "UCxxxxxxxxxxxxxxxxxxxxxx",
    # "@ChannelHandle",
]

MAX_VIDEOS_PER_CHANNEL = 100
YOUTUBE_CACHE_TTL_HOURS = 12  # sadece videos.list istatistikleri icin (kanal kimligi/uploads playlist'i cok daha uzun sure onbellekte kalir)
RUN_YOUTUBE_CONNECTION_TEST = False  # True yaparsan tek, dusuk maliyetli bir baglanti testi calisir
ENABLE_YOUTUBE_SEARCH = False  # Bu standalone defter manuel kanal toplar; anahtar-kelime kesfi master notebook Faz 5'tedir.

# Google Drive'da kalici sonuc/onbellek (Colab'da calisirken onerilir; yerel calistirmada etkisi yoktur)
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/AVRUPA YAKASI/YoutubeOtomationv0"

print("YouTube Data API key:", youtube_data.describe_credential_state(YOUTUBE_DATA_API_KEY) if "yd" in dir() else "yd modulu henuz yuklenmedi")


In [ ]:
# --- Kalici yol cozumleme (Faz 7): Drive (Colab) / /content (Colab, Drive kapali) / yerel results/ ---
youtube_paths = youtube_data.resolve_project_paths(
    is_colab=IS_COLAB,
    use_google_drive=USE_GOOGLE_DRIVE,
    project_root=PROJECT_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
)
YOUTUBE_DATA_DIR = youtube_paths["youtube_data_dir"]
YOUTUBE_CACHE_DIR = youtube_paths["cache_dir"]
print("YouTube veri dizini:", YOUTUBE_DATA_DIR)
print("YouTube onbellek dizini:", YOUTUBE_CACHE_DIR)
print("Drive mount edildi mi:", youtube_paths["drive_mounted"])


In [ ]:
# --- YouTube toplama (Faz 2-6): kanal cozumleme + baglanti testi + upload gecmisi ---
from datetime import datetime, timezone

youtube_started_at = datetime.now(timezone.utc).isoformat()
youtube_cache = youtube_data.Cache(YOUTUBE_CACHE_DIR)
youtube_quota = youtube_data.QuotaTracker()

youtube_client = None
youtube_connection_test_result = None
youtube_resolved_channels = []
youtube_failed_channels = []
youtube_all_videos = []
youtube_collection_errors = []
youtube_status = "insufficient_data"
youtube_status_reason = "baslatilmadi"

if not YOUTUBE_DATA_API_KEY:
    youtube_status = "insufficient_data"
    youtube_status_reason = "YOUTUBE_DATA_API_KEY yok (Colab Secrets kontrol et)"
    print("YouTube Data API key: missing -- YouTube toplama ATLANDI (temiz atlama).")
else:
    print("YouTube Data API key: configured")
    try:
        youtube_client = youtube_data.build_client(YOUTUBE_DATA_API_KEY)
    except Exception as exc:
        youtube_status = "failed"
        youtube_status_reason = "istemci olusturulamadi: " + youtube_data.sanitize_error_text(exc, YOUTUBE_DATA_API_KEY)
        youtube_collection_errors.append({"stage": "build_client", "error": youtube_status_reason})
        print(youtube_status_reason)

    if youtube_client is not None and RUN_YOUTUBE_CONNECTION_TEST:
        test_input = YOUTUBE_CHANNEL_INPUTS[0] if YOUTUBE_CHANNEL_INPUTS else "@GoogleDevelopers"
        youtube_connection_test_result = youtube_data.run_connection_test(youtube_client, YOUTUBE_DATA_API_KEY, test_input)
        print("Baglanti testi ->", youtube_connection_test_result["status"], "girdi:", test_input)
        if youtube_connection_test_result["status"] == "ok":
            print("  Kanal basligi:", youtube_connection_test_result["channel_title"])
            print("  Kanal ID:", youtube_connection_test_result["channel_id"])
            print("  Uploads playlist mevcut mu:", youtube_connection_test_result["has_uploads_playlist"])
        else:
            print("  Kategori:", youtube_connection_test_result["error_category"])
            print("  Aciklama:", youtube_connection_test_result["explanation"])

    if youtube_client is not None and not YOUTUBE_CHANNEL_INPUTS:
        youtube_status = "insufficient_data"
        youtube_status_reason = "YOUTUBE_CHANNEL_INPUTS bos -- toplanacak kanal yok"
        print(youtube_status_reason)
    elif youtube_client is not None:
        youtube_resolved_channels, youtube_failed_channels = youtube_data.resolve_channels(
            youtube_client, YOUTUBE_CHANNEL_INPUTS, api_key_for_sanitization=YOUTUBE_DATA_API_KEY,
        )
        for failed in youtube_failed_channels:
            youtube_collection_errors.append({"stage": "resolve_channel", "input": failed["input"], "error": failed["error"]})
            print("Kanal cozumlenemedi:", failed["input"], "->", failed["error"])

        for channel_record in youtube_resolved_channels:
            rows, err = youtube_data.collect_channel_uploads(
                youtube_client, youtube_cache, youtube_quota, channel_record,
                MAX_VIDEOS_PER_CHANNEL, channel_cache_ttl_hours=YOUTUBE_CACHE_TTL_HOURS,
                video_stats_ttl_hours=YOUTUBE_CACHE_TTL_HOURS, api_key=YOUTUBE_DATA_API_KEY,
            )
            if err:
                youtube_collection_errors.append({"stage": "collect_uploads", "channel_id": channel_record["channel_id"], "error": err})
                print("Toplama hatasi:", channel_record["channel_id"], "->", err)
            youtube_all_videos.extend(rows)

        youtube_all_videos = youtube_data.deduplicate_videos(youtube_all_videos)

        if youtube_resolved_channels and youtube_all_videos:
            youtube_status = "review_required"
            youtube_status_reason = f"{len(youtube_resolved_channels)} kanal, {len(youtube_all_videos)} video toplandi -- insan onayi gerekli"
        elif youtube_resolved_channels and not youtube_all_videos:
            youtube_status = "insufficient_data"
            youtube_status_reason = "kanallar cozumlendi ama kullanilabilir video toplanamadi"
        else:
            youtube_status = "failed"
            youtube_status_reason = "hicbir kanal cozumlenemedi"

youtube_completed_at = datetime.now(timezone.utc).isoformat()
print("\nYouTube toplama durumu:", youtube_status, "--", youtube_status_reason)
print("Onbellek istatistikleri:", youtube_cache.stats)
print("Kota kullanimi (tahmini):", youtube_quota.to_dict())


In [ ]:
# --- YouTube cikti dosyalari (Faz 8): channels.json, normalized_videos.csv, collection_manifest.json, quota_usage.json, errors.json ---
youtube_channels_out = [
    {
        "input": c["input"], "channel_id": c["channel_id"], "title": c["title"],
        "uploads_playlist_id": c["uploads_playlist_id"],
        "subscriber_count": c["subscriber_count"], "video_count": c["video_count"],
    }
    for c in youtube_resolved_channels
]
youtube_data.write_json(YOUTUBE_DATA_DIR / "channels.json", youtube_channels_out)
youtube_data.write_normalized_videos_csv(YOUTUBE_DATA_DIR / "normalized_videos.csv", youtube_all_videos)

youtube_manifest = youtube_data.build_collection_manifest(
    status=youtube_status,
    status_reason=youtube_status_reason,
    started_at=youtube_started_at,
    completed_at=youtube_completed_at,
    requested_channel_count=len(YOUTUBE_CHANNEL_INPUTS),
    resolved_channel_count=len(youtube_resolved_channels),
    failed_channel_count=len(youtube_failed_channels),
    video_count=len(youtube_all_videos),
    date_range=youtube_data.compute_date_range(youtube_all_videos),
    cache_statistics=youtube_cache.stats,
    connection_test_status=(youtube_connection_test_result["status"] if youtube_connection_test_result else None),
    output_paths={
        "channels_json": YOUTUBE_DATA_DIR / "channels.json",
        "normalized_videos_csv": YOUTUBE_DATA_DIR / "normalized_videos.csv",
        "collection_manifest_json": YOUTUBE_DATA_DIR / "collection_manifest.json",
        "quota_usage_json": YOUTUBE_DATA_DIR / "quota_usage.json",
        "errors_json": YOUTUBE_DATA_DIR / "errors.json",
    },
)
youtube_data.write_json(YOUTUBE_DATA_DIR / "collection_manifest.json", youtube_manifest)
youtube_data.write_json(YOUTUBE_DATA_DIR / "quota_usage.json", youtube_quota.to_dict())
youtube_data.write_json(YOUTUBE_DATA_DIR / "errors.json", youtube_collection_errors)

youtube_enrichment_status = youtube_status  # bolum 5/6/7 icin geriye donuk degisken adi korunuyor
print("YouTube ciktilari yazildi ->", YOUTUBE_DATA_DIR)
print("collection_manifest.json status:", youtube_manifest["status"])


## 5. TMDB + Trends + YouTube sinyallerini birleştirme

TMDB/heuristik skoru, LLM yorumu ve YouTube gerçek metrikleri (`normalized_videos.csv`) burada **yan yana raporlanır**, otomatik olarak tek bir "üretim tahmini" olarak birleştirilmez. Bu üç sinyal seti farklı ölçeklerde ve farklı güvenilirlik seviyelerindedir; birleştirme kararı insan/Notebook 05'teki istatistiksel değerlendirmeye bırakılır. `normalized_videos.csv`, Notebook 05'in gerçek geçmiş veriyle çalışması için ana girdi olacaktır.

## 6. Karşılaştırma Tablosu ve Dışa Aktarım

In [ ]:
df = pd.DataFrame(results)
# Not: her satirin 'status' degeri artik log_result tarafindan ayri ayri belirleniyor;
# burada sadece eksik/NaN status degerlerini guvenli varsayilana tamamliyoruz, hepsini EZMIYORUZ.
if 'status' not in df.columns:
    df['status'] = 'experimental'
else:
    df['status'] = df['status'].fillna('experimental')
display(df)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_path = RESULTS_DIR / '04_trend_opportunity_scoring_sonuclar.csv'
df.to_csv(out_path, index=False)
print('Kaydedildi:', out_path)

## 7. Karar (elle doldurulacak)

Not: heuristik ve LLM tabanlı yaklaşım **birlikte** de kullanılabilir (heuristik ön-filtre, LLM ince skorlama) — bu da bir seçenek olarak değerlendirilebilir. YouTube zenginleştirmesi çalıştıysa (`youtube_enrichment_status == 'ok'`), Notebook 05 için `results/youtube_data/normalized_videos.csv` kullanılabilir durumdadır.

- **Seçilen yaklaşım:** _...
- **Gerekçe:** _...
